## [AI09] 경구약제 이미지 객체 검출(Object Detection) 프로젝트

> Evaluation: 이번 대회에서는 mAP@[0.75:0.95]지표를 사용하여 모델의 성능을 측정합니다.
> Submission File: 예측 결과를 담은 csv 파일을 제출합니다. 아래 포맷을 따라 작성해주세요.

|annotation_id|image_id|category_id|bbox_x|bbox_y|bbox_w|bbox_h|score|
|-------------|--------|-----------|------|------|------|------|-----|
|1|1|1|156|247|211|456|0.91|
|2|1|24|498|40|460|474|0.78|
|3|1|11|579|700|260|473|0.27|
|4|1|69|527|83|398|416|0.27|
|5|3|1|143|236|204|135|0.89|
|6|3|24|512|41|388|432|0.78|
|7|3|11|556|677|257|435|0.20|
...


In [ ]:
import kagglehub

kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
path = kagglehub.competition_download("ai09-level1-project")
print(path)

 30%|███       | 554M/1.79G [00:03<00:09, 149MB/s]


KeyboardInterrupt: 

## Module Packages

In [ ]:
import os
import json
import random
import torch
import numpy as np
from PIL import Image
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset
from torchvision import tv_tensors
from torchvision.transforms import v2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import Counter
import copy
!pip install torchmetrics
from torchmetrics.detection.mean_ap import MeanAveragePrecision

In [ ]:
print(os.listdir(path))

In [ ]:
data_path = os.path.join(path, 'sprint_ai_project1_data')
print(os.listdir(data_path))
print("train_annotations 내부:", os.listdir(os.path.join(data_path, "train_annotations"))[:10])
print("train_images 개수:", len(os.listdir(os.path.join(data_path, "train_images"))))
print("test_images 개수:", len(os.listdir(os.path.join(data_path, "test_images"))))


train_annotations 안에 폴더가 하나 더 있다. 폴더 안에 어떤 식으로 파일이 들어있는지 확인해보도록 한다.

In [ ]:
image_dir = os.path.join(data_path, 'train_images')
ann_dir = os.path.join(data_path, 'train_annotations')
test_dir = os.path.join(data_path, 'test_images')

sample_ann_dir = os.path.join(ann_dir, os.listdir(ann_dir)[0])
print("sample_ann_dir:", sample_ann_dir)
print("is dir:", os.path.isdir(sample_ann_dir))
print("inside:", os.listdir(sample_ann_dir))

json_files = []
for root, dirs, files in os.walk(ann_dir):
    for f in files:
        if f.lower().endswith(".json"):
            json_files.append(os.path.join(root, f))

print("찾은 json 파일 수:", len(json_files))
print("예시:", json_files[:1])

In [ ]:
sample_json = json_files[0]
print("sample_json:", sample_json)

with open(sample_json, "r", encoding="utf-8") as f:
    data = json.load(f)

print("type:", type(data))
if isinstance(data, dict):
    print("keys:", data.keys())

이미지 갯수보다 annotation 갯수가 더 많은 것으로 보아, 하나의 이미지에서 각각의 알약에 대한 annotation이 있는 것으로 보인다.  
각각의 폴더에 하나의 이미지 파일에 대한 json 파일이 있는 것으로 보이며, 각 json 파일 안에는 images, type, annotations, categories에 대한 label 정보가 들어있다.

## Dataset Description
본 프로젝트에서 제공하는 데이터는 COCO 포맷 기반 Object Detection 데이터로 핵심 사항은 다음과 같습니다. (원본 데이터 출처: Ai Hub 경구약제 이미지 데이터)
- 이미지 형식: png
- annotation: COCO JSON
- bbox: [x, y, width, height]
- 학습 데이터 구성
    1. train_images
    2. train_annotations
- 테스트 데이터 구성 : test_images

원본 데이터에서는 모든 image_id, category_id, annotation_id 값이 1로 일괄 설정되어 있었으나, 해당 프로젝트에서 제공되는 데이터는 보다 정확한 관리를 위해 아래와 같이 수정되었습니다.
- image_id: 각 이미지의 파일명(file_name)을 기준으로 중복 없이 고유한 ID가 순차적으로 부여됩니다.
- category_id: 각 JSON 파일 내 첫 번째 이미지에 포함된 dl_idx 값을 사용하여 정수형으로 할당됩니다.
- annotation_id: 각 어노테이션에 대해 순차적으로 증가하는 고유 ID가 부여됩니다. 단, 어노테이션이 유효한 경우에만 적용됩니다. 여기서 "존재"와 "유효"의 기준은 다음과 같습니다:
    1. 존재: 어노테이션 항목에 "bbox"라는 키가 실제로 있어야 하며, 값이 비어 있지 않아야 합니다. "bbox" 키가 없거나 비어 있으면 해당 어노테이션은 제외됩니다.
    2. 유효: "bbox"의 값은 리스트 형태이며, 리스트의 길이가 정확히 4여야 합니다. 이는 [x, y, width, height] 형식을 의미하며, 이 조건을 만족해야 어노테이션에 ID가 부여됩니다.

In [ ]:
class COCODataset(Dataset):
    def __init__(self, image_dir, ann_dir, transforms=None):
        self.image_dir = image_dir
        self.ann_dir = ann_dir
        self.transforms = transforms

        #모든 json 파일 수집
        self.json_files = []
        for root, dirs, files in os.walk(ann_dir):
            for f in files:
                if f.lower().endswith('.json'):
                    self.json_files.append(os.path.join(root, f))

        self.json_files.sort()

        #전체 category id 수집
        category_ids = set()
        for json_file in self.json_files:
            with open(json_file, 'r', encoding='utf-8') as f:
                coco = json.load(f)
            for cat in coco['categories']:
                category_ids.add(cat['id'])

        category_ids = sorted(category_ids)

        self.cat_id_to_label = {cat_id: i + 1 for i, cat_id in enumerate(category_ids)}
        self.label_to_cat_id = {i + 1: cat_id for i, cat_id in enumerate(category_ids)}

    def __len__(self):
        return len(self.json_files)

    def __getitem__(self, idx):
        json_file = self.json_files[idx]

        with open(json_file, 'r', encoding='utf-8') as f:
            coco = json.load(f)

        img_info = coco['images'][0]
        image_id = img_info['id']
        file_name = img_info['file_name']

        img_path = os.path.join(self.image_dir, file_name)
        image = Image.open(img_path).convert('RGB')
        w_img, h_img = image.size

        boxes = []
        labels = []
        areas = []
        iscrowd = []

        for ann in coco['annotations']:
            if 'bbox' not in ann or not ann['bbox'] or len(ann['bbox']) != 4:
                continue

            x, y, w, h = ann['bbox']

            if w <= 0 or h <= 0:
                continue

            if w > w_img or h > h_img:
                continue

            x_min = x
            y_min = y
            x_max = x + w
            y_max = y + h

            boxes.append([x_min, y_min, x_max, y_max])
            labels.append(self.cat_id_to_label[ann['category_id']])
            areas.append(ann.get('area', w * h))
            iscrowd.append(ann.get('iscrowd', 0))

        boxes = torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        areas = torch.as_tensor(areas, dtype=torch.float32)
        iscrowd = torch.as_tensor(iscrowd, dtype=torch.int64)

        boxes = tv_tensors.BoundingBoxes(
            boxes,
            format='XYXY',
            canvas_size=(h_img, w_img),
        )
        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': torch.tensor([image_id]),
            'area': areas,
            'iscrowd': iscrowd
        }

        if self.transforms is not None:
            image, target = self.transforms(image, target)
        else:
            image = tv_tensors.Image(image)
            image = v2.ToDtype(torch.float32, scale=True)(image)
            image = v2.ToPureTensor()(image)

        return image, target

In [ ]:
class TestDataset(Dataset):
    def __init__(self, test_dir, transforms=None):
        self.test_dir = test_dir
        self.transforms = transforms
        self.file_names = sorted([
            f for f in os.listdir(test_dir)
            if f.endswith(".png")
        ])

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, idx):
        file_name = self.file_names[idx]
        img_path = os.path.join(self.test_dir, file_name)

        image = Image.open(img_path).convert("RGB")

        if self.transforms:
            image = self.transforms(image)

        return image, file_name

In [ ]:
# 데이터셋 생성 후 EDA 실시
dataset = COCODataset(
    image_dir=image_dir,
    ann_dir=ann_dir,
    transforms=None
)

In [ ]:
idx = 0
image, target = dataset[idx]

json_file = dataset.json_files[idx]
with open(json_file, 'r', encoding='utf-8') as f:
    coco = json.load(f)

img_info = coco['images'][0]
file_name = img_info['file_name']
img_path = os.path.join(dataset.image_dir, file_name)

print("json file:", json_file)
print("file_name in json:", file_name)
print("img_path:", img_path)
print("image size from tensor:", image.shape)   # [C, H, W]
print("boxes:", target['boxes'])
print("labels:", target['labels'])
print("image width/height from json:", img_info.get('width'), img_info.get('height'))

In [ ]:
def visualize_sample(dataset, idx):
    image, target = dataset[idx]

    # tensor -> numpy
    img = image.permute(1, 2, 0).cpu().numpy()

    fig, ax = plt.subplots(1, figsize=(10, 8))
    ax.imshow(img)

    boxes = target['boxes']
    labels = target['labels']

    for box, label in zip(boxes, labels):
        x_min, y_min, x_max, y_max = box.tolist()
        w = x_max - x_min
        h = y_max - y_min

        rect = patches.Rectangle(
            (x_min, y_min), w, h,
            linewidth=2,
            edgecolor='red',
            facecolor='none'
        )
        ax.add_patch(rect)

        class_name = dataset.label_to_cat_id[label.item()]
        ax.text(
            x_min, y_min - 5,
            class_name,
            color='white',
            fontsize=10,
            bbox=dict(facecolor='red', alpha=0.7)
        )

    ax.set_title(f"Sample index: {idx}")
    plt.axis('off')
    plt.show()

In [ ]:
for idx in random.sample(range(len(dataset)), 5):
    visualize_sample(dataset, idx)

In [ ]:
def plot_bbox_size_distribution(dataset):
    widths = []
    heights = []
    areas = []

    for i in range(len(dataset)):
        _, target = dataset[i]
        boxes = target['boxes']

        for box in boxes:
            x_min, y_min, x_max, y_max = box.tolist()
            w = x_max - x_min
            h = y_max - y_min
            widths.append(w)
            heights.append(h)
            areas.append(w * h)

    plt.figure(figsize=(8, 5))
    plt.hist(widths, bins=30, alpha=0.7, label='Width')
    plt.hist(heights, bins=30, alpha=0.7, label='Height')
    plt.legend()
    plt.title("BBox Width / Height Distribution")
    plt.xlabel("Pixels")
    plt.ylabel("Frequency")
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.hist(areas, bins=30)
    plt.title("BBox Area Distribution")
    plt.xlabel("Area")
    plt.ylabel("Frequency")
    plt.show()

plot_bbox_size_distribution(dataset)

In [ ]:
def plot_aspect_ratio_distribution(dataset):
    aspect_ratios = []

    for i in range(len(dataset)):
        _, target = dataset[i]
        boxes = target['boxes']

        for box in boxes:
            x_min, y_min, x_max, y_max = box.tolist()
            w = x_max - x_min
            h = y_max - y_min

            if h > 0:
                aspect_ratios.append(w / h)

    plt.figure(figsize=(8, 5))
    plt.hist(aspect_ratios, bins=30)
    plt.title("BBox Aspect Ratio Distribution")
    plt.xlabel("Width / Height")
    plt.ylabel("Frequency")
    plt.show()

plot_aspect_ratio_distribution(dataset)

In [ ]:
def class_distribution(dataset):
    counter = Counter()

    for i in range(len(dataset)):
        _, target = dataset[i]
        labels = target['labels'].tolist()
        counter.update(labels)

    return counter

counter = class_distribution(dataset)

total = sum(counter.values())
print(f'Total count: {total}')

for label, count in sorted(counter.items()):
    print(f'Class {label}: {count}')

In [ ]:
sorted_items = sorted(counter.items(), key=lambda x: x[1], reverse=True)
labels = [str(x[0]) for x in sorted_items]
counts = [x[1] for x in sorted_items]

plt.figure(figsize=(10, 8))

bars = plt.barh(labels, counts)

# 숫자 라벨 (막대 안쪽에 넣기)
for i, v in enumerate(counts):
    plt.text(v * 0.98, i, str(v),
             va='center', ha='right', fontsize=9, color='white')

# 디자인 정리
plt.gca().invert_yaxis()
plt.xlabel("Count")
plt.title("Class Distribution", fontsize=14, weight='bold')

# 테두리 제거 (핵심 깔끔 포인트)
for spine in ["top", "right", "left"]:
    plt.gca().spines[spine].set_visible(False)

plt.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
class DetectionTransform:
    def __init__(self, train=True):
        if train:
            self.transforms = v2.Compose([
                v2.ToImage(),

                # 기하 변환: bbox도 같이 변환됨
                v2.RandomHorizontalFlip(p=0.5),
                v2.RandomVerticalFlip(p=0.3),
                v2.RandomRotation(degrees=5),
                v2.RandomZoomOut(fill={tv_tensors.Image: (0, 0, 0), "others": 0}, side_range=(1.0, 1.1), p=0.3),

                # 광학 변환
                v2.ColorJitter(
                    brightness=0.15,
                    contrast=0.15,
                    saturation=0.05,
                    hue=0.01
                ),
                v2.RandomAutocontrast(p=0.3),
                v2.RandomEqualize(p=0.2),
                v2.RandomApply([v2.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))], p=0.2),

                # bbox 정리
                v2.ClampBoundingBoxes(),
                v2.SanitizeBoundingBoxes(),

                # tensor 변환
                v2.ToDtype(torch.float32, scale=True),
                v2.ToPureTensor(),
            ])
        else:
            self.transforms = v2.Compose([
                v2.ToImage(),
                v2.ClampBoundingBoxes(),
                v2.SanitizeBoundingBoxes(),
                v2.ToDtype(torch.float32, scale=True),
                v2.ToPureTensor(),
            ])

    def __call__(self, image, target):
        image, target = self.transforms(image, target)
        return image, target

In [ ]:
train_full_dataset = COCODataset(
    image_dir=image_dir,
    ann_dir=ann_dir,
    transforms=DetectionTransform(train=True)
)

val_full_dataset = COCODataset(
    image_dir=image_dir,
    ann_dir=ann_dir,
    transforms=DetectionTransform(train=False)
)

In [ ]:
indices = list(range(len(train_full_dataset)))
random.seed(42)
random.shuffle(indices)

val_ratio = 0.2
split = int(len(indices) * (1 - val_ratio))

train_indices = indices[:split]
val_indices = indices[split:]

# 클래스별 weight
class_weights = {cls: 1 / count for cls, count in counter.items()}

# valid train 샘플만 따로 구성
valid_train_indices = []
sample_weights = []

for idx in train_indices:
    _, target = train_full_dataset[idx]
    labels = target["labels"]

    if labels.numel() == 0:
        continue

    weight = max(class_weights[int(l)] for l in labels)
    valid_train_indices.append(idx)
    sample_weights.append(weight)

train_dataset = Subset(train_full_dataset, valid_train_indices)
val_dataset = Subset(val_full_dataset, val_indices)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

In [ ]:
test_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(dtype=torch.float32, scale=True),
    v2.ToPureTensor(),
])

In [ ]:
test_dataset = TestDataset(
    test_dir=test_dir,
    transforms=test_transform
)

In [ ]:
def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    sampler=sampler,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn
)

## Baseline Model(Faster R-CNN)

In [ ]:
def get_faster_rcnn_model(num_classes):
    # COCO pretrained Faster R-CNN
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
        pretrained=True,
        weights="DEFAULT",
    )

    # 기존 classifier 입력 feature 수
    in_features = model.roi_heads.box_predictor.cls_score.in_features

    # 3. num_classes에 맞게 교체 (background 포함 57개)
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    return model

num_classes = 57
model = get_faster_rcnn_model(num_classes)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
model.to(device)
params = [p for p in model.parameters() if p.requires_grad]

optimizer = torch.optim.SGD(
    params,
    lr=0.005,
    momentum=0.9,
    weight_decay=0.0005
)

lr_scheduler = None

scaler = torch.amp.GradScaler('cuda')

print(device)

In [ ]:
def train_one_epoch(model, dataloader, optimizer, device, scaler):
    model.train()
    total_loss = 0.0
    use_amp = device.type == "cuda"

    for images, targets in dataloader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=use_amp):
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

        scaler.scale(losses).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += losses.item()


    return total_loss / len(dataloader)

In [ ]:
@torch.no_grad()
def validate_loss(model, dataloader, device):
    model.train()  # loss 계산용
    total_loss = 0

    for images, targets in dataloader:
        images = [img.to(device) for img in images]
        targets = [
            {k: v.to(device) for k, v in t.items()}
            for t in targets
        ]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        total_loss += losses.item()

    avg_loss = total_loss / len(dataloader)
    return avg_loss

In [ ]:
@torch.no_grad()
def evaluate_map(model, dataloader, device):
    model.eval()

    metric = MeanAveragePrecision(
        box_format="xyxy",
        iou_type="bbox",
        iou_thresholds=[0.75, 0.80, 0.85, 0.90, 0.95]
    ).to(device)

    for images, targets in dataloader:
        images = [img.to(device) for img in images]

        outputs = model(images)

        preds = []
        gts = []

        for output, target in zip(outputs, targets):
            preds.append({
                "boxes": output["boxes"].to(device),
                "scores": output["scores"].to(device),
                "labels": output["labels"].to(device),
            })

            gts.append({
                "boxes": target["boxes"].to(device),
                "labels": target["labels"].to(device),
            })

        metric.update(preds, gts)

    results = metric.compute()

    return {
        "mAP_75": results["map_75"].item(),         # IoU = 0.75 한 점
        "mAP_75_95": results["map"].item(),         # IoU = 0.75~0.95 평균
    }

In [ ]:
import copy
import torch

def train_with_early_stopping(
    model,
    train_loader,
    val_loader,
    optimizer,
    lr_scheduler,
    device,
    num_epochs=40,
    patience=5,
    min_epochs=10,
    metric_min_delta=1e-4,
    loss_min_delta=1e-3,
    save_path="best_fasterrcnn.pth"
):
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    best_metric = -1.0
    best_val_loss = float("inf")
    best_epoch = -1
    early_stop_counter = 0
    best_model_wts = copy.deepcopy(model.state_dict())

    train_losses = []
    val_losses = []
    map75_list = []
    map7595_list = []

    for epoch in range(num_epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, device, scaler)
        val_loss = validate_loss(model, val_loader, device)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        print(f"Epoch [{epoch+1}/{num_epochs}] "
              f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        metrics = evaluate_map(model, val_loader, device)
        map75 = metrics["mAP_75"]
        map7595 = metrics["mAP_75_95"]

        map75_list.append(map75)
        map7595_list.append(map7595)

        print(f"mAP@75: {map75:.4f} | mAP@75:95: {map7595:.4f}")

        # 개선 여부 먼저 판단
        metric_improved = map7595 > best_metric + metric_min_delta
        loss_improved = val_loss < best_val_loss - loss_min_delta

        # scheduler
        if lr_scheduler is not None:
            if isinstance(lr_scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                lr_scheduler.step(map7595)   # scheduler는 mode='max'로 생성해야 함
            else:
                lr_scheduler.step()

        # best model 저장은 metric 기준
        if metric_improved:
            best_metric = map7595
            best_epoch = epoch + 1
            best_model_wts = copy.deepcopy(model.state_dict())

            torch.save({
                "epoch": best_epoch,
                "model_state_dict": best_model_wts,
                "optimizer_state_dict": optimizer.state_dict(),
                "best_mAP_75_95": best_metric,
                "best_val_loss": val_loss,
            }, save_path)

            print(f"  -> Best model saved at epoch {best_epoch}")

        # early stopping용 best loss 갱신
        if loss_improved:
            best_val_loss = val_loss

        # warmup 구간
        if epoch + 1 < min_epochs:
            print(f"  -> Warmup epoch ({epoch+1}/{min_epochs}), early stopping off")
            continue

        # 둘 중 하나라도 좋아지면 계속
        if metric_improved or loss_improved:
            early_stop_counter = 0
            print("  -> Improvement detected")
        else:
            early_stop_counter += 1
            print(f"  -> No improvement ({early_stop_counter}/{patience})")

        if early_stop_counter >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

    model.load_state_dict(best_model_wts)

    print(f"\nBest Epoch: {best_epoch}")
    print(f"Best mAP@75:95: {best_metric:.4f}")
    print(f"Best Val Loss observed: {best_val_loss:.4f}")

    return model, {
        "best_epoch": best_epoch,
        "best_mAP_75_95": best_metric,
        "best_val_loss": best_val_loss,
        "train_losses": train_losses,
        "val_losses": val_losses,
        "mAP_75": map75_list,
        "mAP_75_95": map7595_list,
    }

In [ ]:
model = get_faster_rcnn_model(num_classes)
model.to(device)

params = [p for p in model.parameters() if p.requires_grad]

optimizer = torch.optim.SGD(
    params,
    lr=0.005,
    momentum=0.9,
    weight_decay=0.0005
)

lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.1,
    patience=3,          # 3 epoch 정체 시 감소
)

scaler = torch.amp.GradScaler('cuda')

model, history = train_with_early_stopping(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    lr_scheduler=lr_scheduler,
    device=device,
    num_epochs=40,
    patience=5,
    save_path="best_fasterrcnn.pth"
)

print("Best epoch:", history["best_epoch"])
print("Best val loss:", history["best_val_loss"])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 마지막 모델 저장
torch.save(model.state_dict(), "/content/drive/MyDrive/last_model.pth")

# 전체 checkpoint 저장 (추천)
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'history': history
}, "/content/drive/MyDrive/fasterrcnn_checkpoint.pth")

print("Best epoch:", history["best_epoch"])
print("Best val loss:", history["best_val_loss"])

In [ ]:
@torch.no_grad()
def predict(model, dataloader, device, score_thresh=0.5):
    model.eval()
    results = []

    for images, _ in dataloader:
        images = [img.to(device) for img in images]
        outputs = model(images)

        for output in outputs:
            keep = output["scores"] >= score_thresh

            results.append({
                "boxes": output["boxes"][keep].cpu(),
                "scores": output["scores"][keep].cpu(),
                "labels": output["labels"][keep].cpu(),
            })

    return results

In [ ]:
import matplotlib.pyplot as plt

def visualize_predictions(dataset, predictions, num_samples=3):
    for i in range(num_samples):
        img, _ = dataset[i]
        pred = predictions[i]

        img = img.permute(1, 2, 0).cpu().numpy()

        boxes = pred["boxes"]
        labels = pred["labels"]

        fig, ax = plt.subplots(1, figsize=(6, 6))
        ax.imshow(img)

        for box in boxes:
            x1, y1, x2, y2 = box
            rect = plt.Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                fill=False,
                edgecolor="red",
                linewidth=2
            )
            ax.add_patch(rect)

        ax.set_title(f"Sample {i}")
        plt.show()

In [ ]:
# 1. mAP 평가
metrics = evaluate_map(model, val_loader, device)
print(metrics)

# 2. test 예측
preds = predict(model, test_loader, device, score_thresh=0.5)

# 3. 시각화
visualize_predictions(test_dataset, preds, num_samples=5)

In [ ]:
import pandas as pd

model.eval()

results = []
annotation_id = 1
score_threshold = 0.5

with torch.no_grad():
    for images, image_paths in test_loader:
        images = [img.to(device) for img in images]
        outputs = model(images)

        for img_path, output in zip(image_paths, outputs):
            file_name = os.path.basename(img_path)              # "000123.jpg"
            image_id = int(os.path.splitext(file_name)[0])      # 123

            boxes = output["boxes"].cpu()
            labels = output["labels"].cpu()
            scores = output["scores"].cpu()

            for box, label, score in zip(boxes, labels, scores):
                if score < score_threshold:
                    continue

                x1, y1, x2, y2 = box.tolist()

                x = x1
                y = y1
                w = x2 - x1
                h = y2 - y1

                # 학습용 label -> 원본 category_id로 복원
                original_cat_id = train_full_dataset.label_to_cat_id[int(label.item())]

                results.append({
                    "annotation_id": annotation_id,
                    "image_id": image_id,
                    "category_id": int(original_cat_id),
                    "bbox_x": int(round(x)),
                    "bbox_y": int(round(y)),
                    "bbox_w": int(round(w)),
                    "bbox_h": int(round(h)),
                    "score": round(score.item(), 2)
                })

                annotation_id += 1

df = pd.DataFrame(results)
df.to_csv("/content/drive/MyDrive/submission.csv", index=False)

### YOLO & RTDETR

In [ ]:
!pip install 'rembg[cpu]'

INFO: pip is looking at multiple versions of numba to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 6.2 MB/s eta 0:00:00
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
   

In [ ]:
import kagglehub

kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
import shutil
from pathlib import Path
from google.colab import drive
import kagglehub

!rm -rf /content/drive
drive.mount('/content/drive', force_remount=True)

DRIVE_PATH = Path("/content/drive/MyDrive/ai_project_data/raw")

# ✅ 이미 있으면 다운로드 안함
if DRIVE_PATH.exists():
    print("✅ 이미 Drive에 데이터 있음 → 다운로드 생략")
else:
    print("⬇️ kaggle에서 다운로드 중...")
    path = kagglehub.competition_download("ai09-level1-project")
    print(path)

    print("📦 Drive로 복사 중...")
    shutil.copytree(path, DRIVE_PATH)

    print("✅ Drive 복사 완료")

Mounted at /content/drive
✅ 이미 Drive에 데이터 있음 → 다운로드 생략


In [ ]:
import os
import json
import random
import shutil
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import math
from PIL import Image, ImageEnhance
import yaml

random.seed(42)

# =========================
# 0. 경로 설정
# =========================
BASE_DIR = Path("/content/drive/MyDrive/ai_project_data/raw/sprint_ai_project1_data")

TRAIN_IMG_DIR = BASE_DIR / "train_images"
TRAIN_ANN_DIR = BASE_DIR / "train_annotations"
TEST_IMG_DIR  = BASE_DIR / "test_images"

OUT_DIR = BASE_DIR / "yolo_dataset"
OUT_TRAIN_IMG = OUT_DIR / "images" / "train"
OUT_VAL_IMG   = OUT_DIR / "images" / "val"
OUT_TEST_IMG  = OUT_DIR / "images" / "test"

OUT_TRAIN_LBL = OUT_DIR / "labels" / "train"
OUT_VAL_LBL   = OUT_DIR / "labels" / "val"
OUT_TEST_LBL  = OUT_DIR / "labels" / "test"

for d in [OUT_TRAIN_IMG, OUT_VAL_IMG, OUT_TEST_IMG, OUT_TRAIN_LBL, OUT_VAL_LBL, OUT_TEST_LBL]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
# =========================
# 0. import / seed / 경로 준비
# =========================
import os
import io
import json
import math
import copy
import random
import shutil
from pathlib import Path
from collections import defaultdict, Counter
from rembg import remove
import numpy as np
import yaml
from PIL import Image, ImageEnhance, ImageFilter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# =========================
# 1. 유틸 함수
# =========================
def clamp(v, lo, hi):
    return max(lo, min(v, hi))

def bbox_xywh_to_xyxy(x, y, w, h):
    return x, y, x + w, y + h

def bbox_xyxy_to_yolo(x1, y1, x2, y2, img_w, img_h):
    cx = (x1 + x2) / 2.0 / img_w
    cy = (y1 + y2) / 2.0 / img_h
    bw = (x2 - x1) / img_w
    bh = (y2 - y1) / img_h
    return cx, cy, bw, bh

def clean_bbox_xywh(
    bbox,
    img_w,
    img_h,
    min_size=2,
    min_area=4,
    min_visible_ratio=0.2,
    max_size_ratio=1.5,
):
    x, y, w, h = bbox

    if w <= 0 or h <= 0:
        return None

    x1, y1, x2, y2 = bbox_xywh_to_xyxy(x, y, w, h)

    box_w = x2 - x1
    box_h = y2 - y1
    if box_w <= 0 or box_h <= 0:
        return None

    if box_w > img_w * max_size_ratio or box_h > img_h * max_size_ratio:
        return None

    orig_area = box_w * box_h
    if orig_area <= 0:
        return None

    x1 = clamp(x1, 0, img_w - 1)
    y1 = clamp(y1, 0, img_h - 1)
    x2 = clamp(x2, 0, img_w - 1)
    y2 = clamp(y2, 0, img_h - 1)

    new_w = x2 - x1
    new_h = y2 - y1
    new_area = new_w * new_h

    if new_w <= 0 or new_h <= 0:
        return None
    if new_w < min_size or new_h < min_size:
        return None
    if new_area < min_area:
        return None

    visible_ratio = new_area / (orig_area + 1e-9)
    if visible_ratio < min_visible_ratio:
        return None

    return [float(x1), float(y1), float(x2), float(y2)]

def load_json_files(ann_dir):
    json_files = []
    for root, _, files in os.walk(ann_dir):
        for f in files:
            if f.lower().endswith(".json"):
                json_files.append(os.path.join(root, f))
    json_files.sort()
    return json_files

def save_yolo_label(label_path, anns, img_w, img_h, cat_id_to_yolo):
    lines = []
    for ann in anns:
        cat_id = ann["category_id"]
        if cat_id not in cat_id_to_yolo:
            continue

        x1, y1, x2, y2 = ann["bbox_xyxy"]
        x1 = max(0, min(x1, img_w - 1))
        y1 = max(0, min(y1, img_h - 1))
        x2 = max(0, min(x2, img_w - 1))
        y2 = max(0, min(y2, img_h - 1))

        if x2 <= x1 or y2 <= y1:
            continue

        cx, cy, bw, bh = bbox_xyxy_to_yolo(x1, y1, x2, y2, img_w, img_h)

        cx = max(0.0, min(cx, 1.0))
        cy = max(0.0, min(cy, 1.0))
        bw = max(1e-6, min(bw, 1.0))
        bh = max(1e-6, min(bh, 1.0))

        lines.append(f"{cat_id_to_yolo[cat_id]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

    with open(label_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

def box_iou_xyxy(box1, box2):
    ax1, ay1, ax2, ay2 = box1
    bx1, by1, bx2, by2 = box2

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    iw = max(0, inter_x2 - inter_x1)
    ih = max(0, inter_y2 - inter_y1)
    inter = iw * ih

    area1 = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area2 = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area1 + area2 - inter + 1e-9
    return inter / union

def non_overlapping_position(existing_boxes, patch_w, patch_h, img_w, img_h, tries=50, max_iou=0.05):
    if patch_w >= img_w or patch_h >= img_h:
        return None

    for _ in range(tries):
        x1 = random.randint(0, max(0, img_w - patch_w))
        y1 = random.randint(0, max(0, img_h - patch_h))
        cand = [x1, y1, x1 + patch_w, y1 + patch_h]

        ok = True
        for b in existing_boxes:
            if box_iou_xyxy(cand, b) > max_iou:
                ok = False
                break

        if ok:
            return cand

    return None

def paste_patch(bg_img, patch, x1, y1):
    bg_img.paste(patch, (x1, y1))
    return bg_img

# image cache
_image_cache = {}

def load_image_cached(img_path):
    key = str(img_path)
    if key not in _image_cache:
        _image_cache[key] = Image.open(img_path).convert("RGB")
    return _image_cache[key]

def clone_annotations(anns):
    return [
        {
            "category_id": ann["category_id"],
            "bbox_xyxy": ann["bbox_xyxy"][:]
        }
        for ann in anns
    ]
# =========================
# rembg synthetic용 유틸
# =========================
def crop_with_rembg(
    img,
    bbox_xyxy,
    pad_ratio=0.08,
    alpha_matting=False,
    post_process_mask=True,
    min_alpha_pixels=100,
):
    """
    img: PIL RGB image
    bbox_xyxy: [x1, y1, x2, y2]
    return: RGBA patch or None
    """
    img_w, img_h = img.size
    x1, y1, x2, y2 = bbox_xyxy

    bw = x2 - x1
    bh = y2 - y1
    if bw <= 1 or bh <= 1:
        return None

    pad_w = int(bw * pad_ratio)
    pad_h = int(bh * pad_ratio)

    x1 = max(0, int(x1 - pad_w))
    y1 = max(0, int(y1 - pad_h))
    x2 = min(img_w, int(x2 + pad_w))
    y2 = min(img_h, int(y2 + pad_h))

    if x2 <= x1 or y2 <= y1:
        return None

    crop = img.crop((x1, y1, x2, y2)).convert("RGBA")

    cutout = remove(
        crop,
        alpha_matting=alpha_matting,
        post_process_mask=post_process_mask,
    )

    if not isinstance(cutout, Image.Image):
        cutout = Image.open(io.BytesIO(cutout)).convert("RGBA")
    else:
        cutout = cutout.convert("RGBA")

    alpha = np.array(cutout.split()[-1])
    if (alpha > 0).sum() < min_alpha_pixels:
        return None

    return cutout


def tight_bbox_from_alpha(rgba_img, alpha_thr=8):
    alpha = np.array(rgba_img.split()[-1])
    ys, xs = np.where(alpha > alpha_thr)

    if len(xs) == 0 or len(ys) == 0:
        return None

    x1 = int(xs.min())
    y1 = int(ys.min())
    x2 = int(xs.max()) + 1
    y2 = int(ys.max()) + 1

    if x2 <= x1 or y2 <= y1:
        return None

    return [x1, y1, x2, y2]


def trim_transparent_border(rgba_img, alpha_thr=8):
    box = tight_bbox_from_alpha(rgba_img, alpha_thr=alpha_thr)
    if box is None:
        return None, None

    x1, y1, x2, y2 = box
    trimmed = rgba_img.crop((x1, y1, x2, y2))
    inner = [0, 0, x2 - x1, y2 - y1]
    return trimmed, inner


def paste_rgba(bg_rgb, patch_rgba, x1, y1):
    bg_rgba = bg_rgb.convert("RGBA")
    bg_rgba.alpha_composite(patch_rgba, dest=(x1, y1))
    return bg_rgba.convert("RGB")


def random_background(img_w, img_h, mode="white"):
    if mode == "white":
        return Image.new("RGB", (img_w, img_h), (255, 255, 255))
    elif mode == "gray":
        g = random.randint(220, 250)
        return Image.new("RGB", (img_w, img_h), (g, g, g))
    elif mode == "light_color":
        color = tuple(random.randint(220, 255) for _ in range(3))
        return Image.new("RGB", (img_w, img_h), color)
    elif mode == "noise":
        arr = np.random.randint(230, 256, (img_h, img_w, 3), dtype=np.uint8)
        return Image.fromarray(arr)
    else:
        return Image.new("RGB", (img_w, img_h), (255, 255, 255))


def build_rembg_object_bank(
    train_records,
    train_img_dir,
    rare_classes=None,
    alpha_matting=False,
    post_process_mask=True,
):
    bank = defaultdict(list)

    for rec in train_records:
        img_path = train_img_dir / rec["file_name"]
        if not img_path.exists():
            continue

        try:
            img = load_image_cached(img_path)
        except Exception:
            continue

        for ann in rec["annotations"]:
            cls = ann["category_id"]
            if rare_classes is not None and cls not in rare_classes:
                continue

            patch = crop_with_rembg(
                img=img,
                bbox_xyxy=ann["bbox_xyxy"],
                pad_ratio=0.08,
                alpha_matting=alpha_matting,
                post_process_mask=post_process_mask,
                min_alpha_pixels=30,
            )
            if patch is None:
                continue

            patch, inner_bbox = trim_transparent_border(patch, alpha_thr=8)
            if patch is None:
                continue

            pw, ph = patch.size
            if pw < 8 or ph < 8:
                continue

            bank[cls].append({
                "patch": patch.copy(),   # RGBA
                "w": pw,
                "h": ph,
                "inner_bbox": inner_bbox
            })

    return bank


def build_rembg_synthetic_image(
    object_bank,
    img_w=960,
    img_h=960,
    min_objects=1,
    max_objects=4,
    min_obj_ratio=0.03,
    max_obj_ratio=0.16,
    position_tries=60,
    max_iou=0.01,
    bg_mode="white",
    cls_weights=None,
):
    cls_candidates = [c for c in object_bank.keys() if len(object_bank[c]) > 0]
    if len(cls_candidates) == 0:
        return None, []

    if cls_weights is None:
        cls_weights = [1.0 for _ in cls_candidates]

    bg = random_background(img_w, img_h, mode=bg_mode)
    anns = []
    placed_boxes = []

    num_objects = random.randint(min_objects, max_objects)

    for _ in range(num_objects):
        cls = random.choices(cls_candidates, weights=cls_weights, k=1)[0]
        obj = random.choice(object_bank[cls])

        patch = obj["patch"].copy()   # RGBA
        pw, ph = obj["w"], obj["h"]

        scale = random.uniform(0.85, 1.15)
        new_w = max(1, int(pw * scale))
        new_h = max(1, int(ph * scale))

        min_w = int(img_w * min_obj_ratio)
        min_h = int(img_h * min_obj_ratio)
        max_w = int(img_w * max_obj_ratio)
        max_h = int(img_h * max_obj_ratio)

        if new_w < min_w or new_h < min_h:
            s = max(min_w / max(new_w, 1), min_h / max(new_h, 1))
            new_w = max(1, int(new_w * s))
            new_h = max(1, int(new_h * s))

        if new_w > max_w or new_h > max_h:
            s = min(max_w / max(new_w, 1), max_h / max(new_h, 1))
            new_w = max(1, int(new_w * s))
            new_h = max(1, int(new_h * s))

        if random.random() < 0.5:
            patch = patch.transpose(Image.FLIP_LEFT_RIGHT)

        if random.random() < 0.35:
            patch = ImageEnhance.Brightness(patch).enhance(random.uniform(0.92, 1.08))

        if random.random() < 0.35:
            patch = ImageEnhance.Contrast(patch).enhance(random.uniform(1.0, 1.15))

        if random.random() < 0.2:
            patch = patch.filter(ImageFilter.GaussianBlur(radius=0.2))

        patch = patch.resize((new_w, new_h), Image.BILINEAR)

        inner_box = tight_bbox_from_alpha(patch, alpha_thr=8)
        if inner_box is None:
            continue

        pos = non_overlapping_position(
            placed_boxes,
            patch_w=new_w,
            patch_h=new_h,
            img_w=img_w,
            img_h=img_h,
            tries=position_tries,
            max_iou=max_iou,
        )
        if pos is None:
            continue

        x1, y1, _, _ = pos
        bg = paste_rgba(bg, patch, x1, y1)

        ix1, iy1, ix2, iy2 = inner_box
        ann_box = [x1 + ix1, y1 + iy1, x1 + ix2, y1 + iy2]

        anns.append({
            "category_id": cls,
            "bbox_xyxy": ann_box
        })
        placed_boxes.append(ann_box)

    return bg, anns
# =========================
# 2. JSON 파싱 -> 이미지 단위 정리
# =========================
json_files = load_json_files(TRAIN_ANN_DIR)
print(f"json files: {len(json_files)}")

image_records = []
used_category_ids = set()
cat_id_to_name = {}

for json_path in json_files:
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if "images" not in data or len(data["images"]) == 0:
        continue

    image_info = data["images"][0]
    img_file = image_info["file_name"]
    img_w = image_info["width"]
    img_h = image_info["height"]

    anns = data.get("annotations", [])
    cats = data.get("categories", [])

    for c in cats:
        cat_id_to_name[c["id"]] = c.get("name", str(c["id"]))

    cleaned_anns = []
    for ann in anns:
        if "bbox" not in ann or "category_id" not in ann:
            continue

        cleaned = clean_bbox_xywh(ann["bbox"], img_w, img_h)
        if cleaned is None:
            continue

        cls = ann["category_id"]
        used_category_ids.add(cls)

        cleaned_anns.append({
            "category_id": cls,
            "bbox_xyxy": cleaned
        })

    image_records.append({
        "json_path": json_path,
        "file_name": img_file,
        "width": img_w,
        "height": img_h,
        "annotations": cleaned_anns
    })

category_ids = sorted(used_category_ids)
cat_id_to_yolo = {cat_id: idx for idx, cat_id in enumerate(category_ids)}
yolo_to_cat_id = {idx: cat_id for cat_id, idx in cat_id_to_yolo.items()}

print(f"num classes: {len(category_ids)}")
print(f"category mapping: {cat_id_to_yolo}")

# =========================
# 3. train / val split
# =========================
val_ratio = 0.2
MIN_TRAIN_IMAGES_PER_RARE_CLASS = 2
MIN_VAL_IMAGES_PER_RARE_CLASS = 1
RARE_IMAGE_THRESHOLD = 8

class_to_indices = defaultdict(list)
for i, rec in enumerate(image_records):
    cls_set = {ann["category_id"] for ann in rec["annotations"]}
    for cls in cls_set:
        class_to_indices[cls].append(i)

rare_classes_for_split = {
    cls for cls, idxs in class_to_indices.items()
    if len(idxs) <= RARE_IMAGE_THRESHOLD
}

print(f"rare classes for split ({len(rare_classes_for_split)}):",
      sorted(list(rare_classes_for_split))[:20], "...")

num_images = len(image_records)
target_train_size = int(num_images * (1 - val_ratio))
target_val_size = num_images - target_train_size

all_indices = set(range(num_images))
train_idx_set = set()
val_idx_set = set()

# 1) rare class 최소 train/val 확보
for cls in sorted(rare_classes_for_split):
    idxs = class_to_indices[cls][:]
    random.shuffle(idxs)
    total = len(idxs)

    if total == 1:
        need_train, need_val = 1, 0
    elif total == 2:
        need_train, need_val = 1, 1
    else:
        need_train = min(MIN_TRAIN_IMAGES_PER_RARE_CLASS, total - 1)
        need_val = min(MIN_VAL_IMAGES_PER_RARE_CLASS, total - need_train)

    val_added = 0
    for idx in idxs:
        if val_added >= need_val:
            break
        if idx not in train_idx_set and idx not in val_idx_set:
            val_idx_set.add(idx)
            val_added += 1

    train_added = 0
    for idx in idxs:
        if train_added >= need_train:
            break
        if idx not in train_idx_set and idx not in val_idx_set:
            train_idx_set.add(idx)
            train_added += 1

    current_train = sum(1 for idx in idxs if idx in train_idx_set)
    current_val = sum(1 for idx in idxs if idx in val_idx_set)

    if current_train < need_train:
        for idx in idxs:
            if current_train >= need_train:
                break
            if idx in val_idx_set:
                val_idx_set.remove(idx)
                train_idx_set.add(idx)
                current_train += 1
                current_val -= 1

    if current_val < need_val:
        for idx in idxs:
            if current_val >= need_val:
                break
            if idx in train_idx_set:
                train_after_remove = sum(1 for j in idxs if j in train_idx_set and j != idx)
                if train_after_remove >= need_train:
                    train_idx_set.remove(idx)
                    val_idx_set.add(idx)
                    current_train -= 1
                    current_val += 1
            elif idx not in val_idx_set:
                val_idx_set.add(idx)
                current_val += 1

# 2) 전체 target size 맞추기
remaining_indices = list(all_indices - train_idx_set - val_idx_set)
random.shuffle(remaining_indices)

need_val_more = max(0, target_val_size - len(val_idx_set))
val_fill = remaining_indices[:need_val_more]
val_idx_set.update(val_fill)

remaining_indices = [i for i in remaining_indices if i not in val_idx_set]
train_idx_set.update(remaining_indices)

if len(train_idx_set) < target_train_size:
    movable_from_val = list(val_idx_set)
    random.shuffle(movable_from_val)

    for idx in movable_from_val:
        if len(train_idx_set) >= target_train_size:
            break

        rec = image_records[idx]
        classes_in_img = {ann["category_id"] for ann in rec["annotations"]}

        can_move = True
        for cls in classes_in_img:
            idxs = class_to_indices[cls]
            total = len(idxs)

            if cls in rare_classes_for_split:
                if total == 1:
                    min_val_needed = 0
                elif total == 2:
                    min_val_needed = 1
                else:
                    min_val_needed = MIN_VAL_IMAGES_PER_RARE_CLASS
            else:
                min_val_needed = 0

            current_val_count = sum(1 for j in idxs if j in val_idx_set)
            if current_val_count - 1 < min_val_needed:
                can_move = False
                break

        if can_move:
            val_idx_set.remove(idx)
            train_idx_set.add(idx)

if len(train_idx_set) > target_train_size:
    movable_from_train = list(train_idx_set)
    random.shuffle(movable_from_train)

    for idx in movable_from_train:
        if len(train_idx_set) <= target_train_size:
            break

        rec = image_records[idx]
        classes_in_img = {ann["category_id"] for ann in rec["annotations"]}

        can_move = True
        for cls in classes_in_img:
            idxs = class_to_indices[cls]
            total = len(idxs)

            if cls in rare_classes_for_split:
                if total == 1:
                    min_train_needed = 1
                elif total == 2:
                    min_train_needed = 1
                else:
                    min_train_needed = MIN_TRAIN_IMAGES_PER_RARE_CLASS
            else:
                min_train_needed = 0

            current_train_count = sum(1 for j in idxs if j in train_idx_set)
            if current_train_count - 1 < min_train_needed:
                can_move = False
                break

        if can_move:
            train_idx_set.remove(idx)
            val_idx_set.add(idx)

train_indices = sorted(train_idx_set)
val_indices = sorted(val_idx_set)

assert len(set(train_indices) & set(val_indices)) == 0
assert len(train_indices) + len(val_indices) == num_images

train_records = [image_records[i] for i in train_indices]
val_records = [image_records[i] for i in val_indices]

print(f"train: {len(train_records)}")
print(f"val: {len(val_records)}")

train_class_image_freq = Counter()
val_class_image_freq = Counter()

for rec in train_records:
    for cls in {ann["category_id"] for ann in rec["annotations"]}:
        train_class_image_freq[cls] += 1

for rec in val_records:
    for cls in {ann["category_id"] for ann in rec["annotations"]}:
        val_class_image_freq[cls] += 1

print("rare class coverage check:")
for cls in sorted(rare_classes_for_split):
    total_img_cnt = len(class_to_indices[cls])
    train_img_cnt = train_class_image_freq[cls]
    val_img_cnt = val_class_image_freq[cls]

    if total_img_cnt == 1:
        target_train, target_val = 1, 0
    elif total_img_cnt == 2:
        target_train, target_val = 1, 1
    else:
        target_train = min(MIN_TRAIN_IMAGES_PER_RARE_CLASS, total_img_cnt - 1)
        target_val = min(MIN_VAL_IMAGES_PER_RARE_CLASS, total_img_cnt - target_train)

    print(
        f"  class {cls}: total={total_img_cnt}, "
        f"train={train_img_cnt}, val={val_img_cnt}, "
        f"target_train>={target_train}, target_val>={target_val}"
    )

# =========================
# 4. 클래스 빈도 계산
#    - bbox freq: rare object bank 판단용
#    - image freq: oversampling repeat 판단용
# =========================
train_class_bbox_freq = Counter()
train_class_image_freq = Counter()

for rec in train_records:
    cls_set = set()
    for ann in rec["annotations"]:
        cls = ann["category_id"]
        train_class_bbox_freq[cls] += 1
        cls_set.add(cls)
    for cls in cls_set:
        train_class_image_freq[cls] += 1

if len(train_class_bbox_freq) == 0:
    raise ValueError("train_class_bbox_freq is empty. Check annotation cleaning or split logic.")

print(
    "train class bbox freq min/max:",
    min(train_class_bbox_freq.values()),
    max(train_class_bbox_freq.values())
)
print(
    "train class image freq min/max:",
    min(train_class_image_freq.values()),
    max(train_class_image_freq.values())
)

# =========================
# 5. oversampling repeat factor 계산
#    image frequency 기준
# =========================
TARGET_COUNT = 10
MAX_REPEAT = 4

oversampled_train_records = []
repeat_stats = Counter()

for rec in train_records:
    if len(rec["annotations"]) == 0:
        continue

    cls_in_img = list({ann["category_id"] for ann in rec["annotations"]})
    min_cls_count = min(train_class_image_freq[c] for c in cls_in_img)

    base_repeat = math.sqrt(TARGET_COUNT / max(1, min_cls_count))
    repeat = np.random.poisson(base_repeat)
    repeat = max(1, min(MAX_REPEAT, repeat))

    if min_cls_count <= 2:
        repeat = min(MAX_REPEAT, repeat + 1)

    repeat_stats[repeat] += 1
    for _ in range(repeat):
        oversampled_train_records.append(rec)

print("repeat distribution:", dict(repeat_stats))
print("oversampled train images:", len(oversampled_train_records))

# =========================
# 6. rare object crop bank
# =========================
RARE_THRESHOLD = 10
rare_classes = {
    cls for cls, cnt in train_class_bbox_freq.items()
    if cnt <= RARE_THRESHOLD
}
print(f"rare classes ({len(rare_classes)}):", sorted(list(rare_classes))[:20], "...")

object_bank = defaultdict(list)

for rec in train_records:
    img_path = TRAIN_IMG_DIR / rec["file_name"]
    if not img_path.exists():
        continue

    try:
        img = load_image_cached(img_path)
    except Exception:
        continue

    for ann in rec["annotations"]:
        cls = ann["category_id"]
        if cls not in rare_classes:
            continue

        x1, y1, x2, y2 = map(int, ann["bbox_xyxy"])
        if x2 <= x1 or y2 <= y1:
            continue

        patch = img.crop((x1, y1, x2, y2))
        pw, ph = patch.size
        if pw < 10 or ph < 10:
            continue

        object_bank[cls].append({
            "patch": patch.copy(),
            "w": pw,
            "h": ph
        })

print("object bank sizes:")
for cls in sorted(object_bank.keys())[:10]:
    print(f"  class {cls}: {len(object_bank[cls])}")

# =========================
# 6-1. rembg object bank 만들기
# =========================
REMBG_USE_RARE_ONLY = True

rembg_source_classes = rare_classes if REMBG_USE_RARE_ONLY else None

rembg_bank = build_rembg_object_bank(
    train_records=train_records,
    train_img_dir=TRAIN_IMG_DIR,
    rare_classes=rembg_source_classes,
    alpha_matting=False,        # 처음엔 False 추천
    post_process_mask=True,
)

print("rembg bank sizes:")
shown = 0
for cls in sorted(rembg_bank.keys()):
    print(f"  class {cls}: {len(rembg_bank[cls])}")
    shown += 1
    if shown >= 10:
        break
# =========================
# 7. train/val 원본 복사 + YOLO 라벨 생성
# =========================
def process_records(records, out_img_dir, out_lbl_dir, copy_images=True, prefix=""):
    valid_count = 0
    for rec in records:
        src_img = TRAIN_IMG_DIR / rec["file_name"]
        if not src_img.exists():
            continue

        stem = Path(rec["file_name"]).stem
        suffix = Path(rec["file_name"]).suffix

        dst_img = out_img_dir / f"{prefix}{stem}{suffix}"
        dst_lbl = out_lbl_dir / f"{prefix}{stem}.txt"

        if copy_images:
            shutil.copy2(src_img, dst_img)

        save_yolo_label(dst_lbl, rec["annotations"], rec["width"], rec["height"], cat_id_to_yolo)
        valid_count += 1
    return valid_count

n_train = process_records(train_records, OUT_TRAIN_IMG, OUT_TRAIN_LBL, copy_images=True, prefix="")
n_val = process_records(val_records, OUT_VAL_IMG, OUT_VAL_LBL, copy_images=True, prefix="")

print("base train copied:", n_train)
print("base val copied:", n_val)

# =========================
# 8. oversampling + copy-paste
# =========================
COPY_PASTE_PER_IMAGE = 1
MAX_PASTE_OBJECTS = 1
POSITION_TRIES = 20
AUG_PREFIX = "aug_"
LOG_EVERY = 100

MIN_OBJ_RATIO = 0.03
MAX_OBJ_RATIO = 0.35
MAX_IOU_FOR_PASTE = 0.05

aug_created = 0

cls_candidates = [c for c in rare_classes if len(object_bank[c]) > 0]
cls_weights = []
for c in cls_candidates:
    freq = train_class_bbox_freq[c]
    cls_weights.append(1.0 / math.sqrt(max(freq, 1)))

for idx, rec in enumerate(oversampled_train_records):
    if idx % LOG_EVERY == 0:
        print(f"[copy-paste] {idx}/{len(oversampled_train_records)}")

    img_path = TRAIN_IMG_DIR / rec["file_name"]
    if not img_path.exists():
        continue

    try:
        base_img = load_image_cached(img_path)
    except Exception:
        continue

    img_w, img_h = rec["width"], rec["height"]
    existing_boxes = [ann["bbox_xyxy"][:] for ann in rec["annotations"]]
    base_anns = clone_annotations(rec["annotations"])

    if len(cls_candidates) == 0:
        continue

    for k in range(COPY_PASTE_PER_IMAGE):
        img_aug = base_img.copy()
        anns_aug = clone_annotations(base_anns)
        boxes_aug = [b[:] for b in existing_boxes]

        cls_in_img = list({ann["category_id"] for ann in rec["annotations"]})
        if len(cls_in_img) == 0:
            continue

        min_cls_count = min(train_class_image_freq[c] for c in cls_in_img)
        num_to_paste = 1 if min_cls_count > 5 else random.choice([1, MAX_PASTE_OBJECTS])

        pasted = 0
        for _ in range(num_to_paste):
            cls = random.choices(cls_candidates, weights=cls_weights, k=1)[0]
            obj = random.choice(object_bank[cls])

            pw, ph = obj["w"], obj["h"]
            patch = obj["patch"].copy()

            scale = random.uniform(0.85, 1.15)
            new_w = int(pw * scale)
            new_h = int(ph * scale)

            min_w = int(img_w * MIN_OBJ_RATIO)
            min_h = int(img_h * MIN_OBJ_RATIO)
            max_w = int(img_w * MAX_OBJ_RATIO)
            max_h = int(img_h * MAX_OBJ_RATIO)

            if new_w < min_w or new_h < min_h:
                continue

            if new_w > max_w or new_h > max_h:
                scale_down = min(max_w / max(new_w, 1), max_h / max(new_h, 1))
                new_w = max(1, int(new_w * scale_down))
                new_h = max(1, int(new_h * scale_down))

            if new_w <= 1 or new_h <= 1:
                continue

            if random.random() < 0.5:
                patch = patch.transpose(Image.FLIP_LEFT_RIGHT)

            if random.random() < 0.4:
                patch = ImageEnhance.Brightness(patch).enhance(random.uniform(0.92, 1.08))

            if random.random() < 0.4:
                patch = ImageEnhance.Contrast(patch).enhance(random.uniform(1.0, 1.2))

            if random.random() < 0.3:
                patch = patch.filter(ImageFilter.GaussianBlur(radius=0.3))

            patch = patch.resize((new_w, new_h), Image.BILINEAR)

            pw2, ph2 = patch.size
            pos = non_overlapping_position(
                boxes_aug, pw2, ph2, img_w, img_h,
                tries=POSITION_TRIES,
                max_iou=MAX_IOU_FOR_PASTE
            )

            if pos is None:
                continue

            x1, y1, _, _ = pos
            x2 = x1 + pw2
            y2 = y1 + ph2

            img_aug = paste_patch(img_aug, patch, x1, y1)

            anns_aug.append({
                "category_id": cls,
                "bbox_xyxy": [x1, y1, x2, y2]
            })
            boxes_aug.append([x1, y1, x2, y2])
            pasted += 1

        if pasted == 0:
            continue

        stem = Path(rec["file_name"]).stem
        suffix = Path(rec["file_name"]).suffix

        out_name = f"{AUG_PREFIX}{idx}_{k}_{stem}{suffix}"
        out_img_path = OUT_TRAIN_IMG / out_name
        out_lbl_path = OUT_TRAIN_LBL / f"{Path(out_name).stem}.txt"

        img_aug.save(out_img_path, optimize=False)
        save_yolo_label(out_lbl_path, anns_aug, img_w, img_h, cat_id_to_yolo)
        aug_created += 1

        if aug_created % LOG_EVERY == 0:
            print(f"  augmented saved: {aug_created}")

print("copy-paste augmented images:", aug_created)

# =========================
# 8-1. rembg blank synthetic 생성
# =========================
REMBG_SYNTH_COUNT = 1000
REMBG_SYNTH_PREFIX = "rembgsyn"
REMBG_BG_MODES = ["white", "gray", "light_color", "noise"]

rembg_cls_candidates = [c for c in rembg_bank.keys() if len(rembg_bank[c]) > 0]
rembg_cls_weights = []

for c in rembg_cls_candidates:
    freq = train_class_bbox_freq.get(c, 1)
    rembg_cls_weights.append(1.0 / math.sqrt(max(freq, 1)))

rembg_created = 0

for i in range(REMBG_SYNTH_COUNT):
    if len(rembg_cls_candidates) == 0:
        break

    bg_mode = random.choice(REMBG_BG_MODES)

    syn_img, syn_anns = build_rembg_synthetic_image(
        object_bank=rembg_bank,
        img_w=960,
        img_h=960,
        min_objects=1,
        max_objects=4,
        min_obj_ratio=0.03,
        max_obj_ratio=0.16,
        position_tries=60,
        max_iou=0.01,
        bg_mode=bg_mode,
        cls_weights=rembg_cls_weights,
    )

    if syn_img is None or len(syn_anns) == 0:
        continue

    out_name = f"{REMBG_SYNTH_PREFIX}_{i:05d}.png"
    out_img_path = OUT_TRAIN_IMG / out_name
    out_lbl_path = OUT_TRAIN_LBL / f"{Path(out_name).stem}.txt"

    syn_img.save(out_img_path)
    save_yolo_label(out_lbl_path, syn_anns, 960, 960, cat_id_to_yolo)

    rembg_created += 1

print("rembg synthetic images:", rembg_created)
# =========================
# 9. test 이미지 복사
# =========================
if TEST_IMG_DIR.exists():
    test_imgs = sorted([
        p for p in TEST_IMG_DIR.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]
    ])
    for p in test_imgs:
        shutil.copy2(p, OUT_TEST_IMG / p.name)
    print("test images copied:", len(test_imgs))

# =========================
# 10. data.yaml 생성
# =========================
names = [cat_id_to_name.get(cat_id, str(cat_id)) for cat_id in category_ids]

data_yaml = {
    "path": str(OUT_DIR),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test" if TEST_IMG_DIR.exists() else "",
    "nc": len(names),
    "names": names,
}

with open(OUT_DIR / "data.yaml", "w", encoding="utf-8") as f:
    yaml.dump(data_yaml, f, allow_unicode=True, sort_keys=False)

print(f"saved data.yaml -> {OUT_DIR / 'data.yaml'}")
print("DONE")

json files: 763
num classes: 56
category mapping: {1900: 0, 2483: 1, 3351: 2, 3483: 3, 3544: 4, 3743: 5, 3832: 6, 4543: 7, 12081: 8, 12247: 9, 12778: 10, 13395: 11, 13900: 12, 16232: 13, 16262: 14, 16548: 15, 16551: 16, 16688: 17, 18147: 18, 18357: 19, 19232: 20, 19552: 21, 19607: 22, 19861: 23, 20014: 24, 20238: 25, 20877: 26, 21325: 27, 21771: 28, 22074: 29, 22347: 30, 22362: 31, 24850: 32, 25367: 33, 25438: 34, 25469: 35, 27733: 36, 27777: 37, 27926: 38, 27993: 39, 28763: 40, 29345: 41, 29451: 42, 29667: 43, 30308: 44, 31863: 45, 31885: 46, 32310: 47, 33009: 48, 33208: 49, 33880: 50, 34597: 51, 35206: 52, 36637: 53, 38162: 54, 41768: 55}
rare classes for split (25): [3544, 3743, 4543, 12081, 12247, 12778, 13395, 16551, 16688, 19552, 19607, 19861, 21771, 22362, 24850, 25438, 27777, 27926, 27993, 29345] ...
train: 610
val: 153
rare class coverage check:
  class 3544: total=6, train=4, val=2, target_train>=2, target_val>=1
  class 3743: total=3, train=2, val=1, target_train>=2, target_

object bank sizes:
  class 1900: 10
  class 2483: 7
  class 3544: 4
  class 3743: 2
  class 4543: 3
  class 12081: 2
  class 12247: 2
  class 12778: 4
  class 13395: 2
  class 16551: 2


  0%|                                               | 0.00/176M [00:00<?, ?B/s]

rembg bank sizes:
  class 1900: 10
  class 2483: 7
  class 3544: 4
  class 3743: 2
  class 4543: 3
  class 12081: 2
  class 12247: 2
  class 12778: 4
  class 13395: 2
  class 16551: 2
base train copied: 610
base val copied: 153
[copy-paste] 0/811
  augmented saved: 100
[copy-paste] 100/811
  augmented saved: 200
[copy-paste] 200/811
  augmented saved: 300
[copy-paste] 300/811
  augmented saved: 400
[copy-paste] 400/811
  augmented saved: 500
[copy-paste] 500/811
  augmented saved: 600
[copy-paste] 600/811
  augmented saved: 700
[copy-paste] 700/811
  augmented saved: 800
[copy-paste] 800/811
copy-paste augmented images: 811
rembg synthetic images: 1000
test images copied: 842
saved data.yaml -> /content/drive/MyDrive/ai_project_data/raw/sprint_ai_project1_data/yolo_dataset/data.yaml
DONE


In [ ]:
from pathlib import Path
import shutil

src = Path("/content/drive/MyDrive/ai_project_data/raw/sprint_ai_project1_data/yolo_dataset")
dst = Path("/content/yolo_dataset")

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)
print("copy done:", dst)

copy done: /content/yolo_dataset


In [ ]:
import random
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import math

# 경로
IMG_DIR = OUT_DIR / "images" / "train"
LBL_DIR = OUT_DIR / "labels" / "train"

NUM_SAMPLES = 40
COLS = 5

img_paths = list(IMG_DIR.glob("*"))
samples = random.sample(img_paths, min(NUM_SAMPLES, len(img_paths)))

def yolo_to_xyxy(cx, cy, w, h, img_w, img_h):
    x1 = (cx - w/2) * img_w
    y1 = (cy - h/2) * img_h
    x2 = (cx + w/2) * img_w
    y2 = (cy + h/2) * img_h
    return int(x1), int(y1), int(x2), int(y2)

rows = math.ceil(len(samples) / COLS)

plt.figure(figsize=(COLS * 4, rows * 4))

for i, img_path in enumerate(samples):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    h, w, _ = img.shape
    label_path = LBL_DIR / (img_path.stem + ".txt")

    if label_path.exists():
        with open(label_path, "r") as f:
            lines = f.readlines()

        for line in lines:
            cls, cx, cy, bw, bh = map(float, line.strip().split())

            x1, y1, x2, y2 = yolo_to_xyxy(cx, cy, bw, bh, w, h)

            # bbox
            cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)

            # class label
            cv2.putText(img, str(int(cls)), (x1, max(0, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

    plt.subplot(rows, COLS, i + 1)
    plt.imshow(img)
    plt.title(img_path.name[:15])  # 이름 너무 길면 잘라서
    plt.axis("off")

plt.tight_layout()
plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
!pip install ultralytics
from ultralytics import YOLO, RTDETR

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
from ultralytics import YOLO

OUT_DIR = BASE_DIR / "yolo_dataset"

model = YOLO("yolov8m.pt")

results = model.train(
    data=str(dst / "data.yaml"),
    epochs=120,
    imgsz=960,
    batch=8,
    device=0,
    workers=4,

    project="runs_yolo",
    name="exp_yolo8m",

    pretrained=True,
    patience=20,

    optimizer="AdamW",
    lr0=5e-4,
    lrf=0.01,
    weight_decay=5e-4,
    cos_lr=True,

    warmup_epochs=5,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,

    box=7.5,
    cls=2.5,

    # color augmentation: 약하게
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.1,

    # geometry augmentation: 약중간
    degrees=0.0,
    translate=0.03,
    scale=0.05,
    shear=0.0,
    perspective=0.0,

    fliplr=0.5,
    flipud=0.5,

    # composition augmentation: 약하게
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
    erasing=0.0,
    cutmix=0.0,
    auto_augment=None,
)

Ultralytics 8.4.32 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=None, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=2.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.0, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.1, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=exp_yolo8m, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mas

In [ ]:
import shutil
from pathlib import Path

SRC = Path("/content/runs/detect/runs_yolo/exp_yolo8m/weights")
DST = Path("/content/drive/MyDrive/ai_project_data/raw/sprint_ai_project1_data/yolo_dataset/yolo_weights")

DST.mkdir(parents=True, exist_ok=True)

shutil.copy(SRC / "best.pt", DST / "best_yolo8m.pt")
shutil.copy(SRC / "last.pt", DST / "last_yolo8m.pt")

print("✅ weights 복사 완료")

✅ weights 복사 완료


In [ ]:
from ultralytics import YOLO

OUT_DIR = BASE_DIR / "yolo_dataset"

model = YOLO("yolo11m.pt")

results = model.train(
    data=str(dst / "data.yaml"),
    epochs=120,
    imgsz=960,
    batch=8,
    device=0,
    workers=4,

    project="runs_yolo",
    name="exp_yolo11m",

    pretrained=True,
    patience=20,

    optimizer="AdamW",
    lr0=4e-4,
    lrf=0.01,
    weight_decay=5e-4,
    cos_lr=True,

    warmup_epochs=5,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,

    box=7.5,
    cls=2.5,

    # color augmentation: 약하게
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.1,

    # geometry augmentation: 약중간
    degrees=0.0,
    translate=0.03,
    scale=0.05,
    shear=0.0,
    perspective=0.0,

    fliplr=0.5,
    flipud=0.5,

    # composition augmentation: 약하게
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
    erasing=0.0,
    cutmix=0.0,
    auto_augment=None,
)

Ultralytics 8.4.32 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=None, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=2.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.0, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.1, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0004, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=exp_yolo11m, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_ma

In [ ]:
import shutil
from pathlib import Path

SRC = Path("/content/runs/detect/runs_yolo/exp_yolo11m/weights")
DST = Path("/content/drive/MyDrive/ai_project_data/raw/sprint_ai_project1_data/yolo_dataset/yolo_weights")

DST.mkdir(parents=True, exist_ok=True)

shutil.copy(SRC / "best.pt", DST / "best_yolo11m.pt")
shutil.copy(SRC / "last.pt", DST / "last_yolo11m.pt")

print("✅ weights 복사 완료")

✅ weights 복사 완료


In [ ]:
from ultralytics import RTDETR

OUT_DIR = BASE_DIR / "yolo_dataset"

model = RTDETR('rtdetr-l.pt')

results = model.train(
    data=str(OUT_DIR / "data.yaml"),
    epochs=120,
    imgsz=960,
    batch=8,
    device=0,
    workers=4,

    project="runs_rtdetr",
    name="exp_rtdetr_l",

    pretrained=True,
    patience=20,

    optimizer="AdamW",
    lr0=4e-4,
    lrf=0.01,
    weight_decay=5e-4,
    cos_lr=True,

    warmup_epochs=5,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,

    box=7.5,
    cls=2.5,

    # color augmentation: 약하게
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.1,

    # geometry augmentation: 약중간
    degrees=0.0,
    translate=0.03,
    scale=0.05,
    shear=0.0,
    perspective=0.0,

    fliplr=0.5,
    flipud=0.5,

    # composition augmentation: 약하게
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
    erasing=0.0,
    cutmix=0.0,
    auto_augment=None,
)

Ultralytics 8.4.32 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=None, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=2.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/ai_project_data/raw/sprint_ai_project1_data/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.0, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.1, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0004, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=exp_rtdetr_l2, nbs=64, nms=F

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/120      23.4G     0.4029      2.672     0.2425          4        960: 100% ━━━━━━━━━━━━ 438/438 5.3it/s 1:23
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 13.5it/s 0.9s
                   all        181        180      0.212      0.142     0.0413     0.0261

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      2/120      23.9G     0.2249      2.502     0.1223         18        960: 0% ──────────── 0/438  0.2s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/120      23.9G     0.1986       2.18    0.09744          2        960: 100% ━━━━━━━━━━━━ 438/438 5.5it/s 1:19
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 13.2it/s 0.9s
                   all        181        180     0.0696      0.879      0.143      0.123

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      3/120      23.9G    0.06685      2.155    0.02448         16        960: 0% ──────────── 0/438  0.2s

...

     28/120      23.9G    0.03141     0.3001    0.01805          4        960: 100% ━━━━━━━━━━━━ 438/438 5.7it/s 1:17
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 13.1it/s 0.9s
                   all        181        180       0.33      0.929      0.405      0.399

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
     29/120      23.9G    0.03993     0.2358    0.02234         16        960: 0% ──────────── 0/438  0.2s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     29/120      23.9G    0.03077     0.2959    0.01796          4        960: 100% ━━━━━━━━━━━━ 438/438 5.7it/s 1:17
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 13.0it/s 0.9s
                   all        181        180      0.277      0.878      0.399      0.395

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
     30/120      23.9G    0.02736     0.2776    0.01541         16        960: 0% ──────────── 0/438  0.2s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     30/120      23.9G    0.03032     0.2985    0.01778          2        960: 100% ━━━━━━━━━━━━ 438/438 5.7it/s 1:17
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 13.2it/s 0.9s
                   all        181        180      0.303      0.965       0.42      0.414
EarlyStopping: Training stopped early as no improvement observed in last 20 epochs. Best results observed at epoch 10, best model saved as best.pt.
To update EarlyStopping(patience=20) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

30 epochs completed in 0.657 hours.
Optimizer stripped from /content/runs/detect/runs_rtdetr/exp_rtdetr_l2/weights/last.pt, 66.5MB
Optimizer stripped from /content/runs/detect/runs_rtdetr/exp_rtdetr_l2/weights/best.pt, 66.5MB

Validating /content/runs/detect/runs_rtdetr/exp_rtdetr_l2/weights/best.pt...
Ultralytics 8.4.32 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000

In [ ]:
import shutil
from pathlib import Path

SRC = Path("/content/runs/detect/runs_rtdetr/exp_rtdetr_l2/weights")
DST = Path("/content/drive/MyDrive/ai_project_data/raw/sprint_ai_project1_data/yolo_dataset/detr_weights")

DST.mkdir(parents=True, exist_ok=True)

shutil.copy(SRC / "best.pt", DST / "best_rtdetr.pt")
shutil.copy(SRC / "last.pt", DST / "last_rtdetr.pt")

print("✅ weights 복사 완료")

✅ weights 복사 완료


##Classifier

In [30]:
import os
import json
import math
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from tqdm import tqdm

# =========================
# 경로 설정
# =========================
BASE_DIR = Path("/content/drive/MyDrive/ai_project_data/raw/sprint_ai_project1_data")

TRAIN_IMG_DIR = BASE_DIR / "train_images"
TRAIN_ANN_DIR = BASE_DIR / "train_annotations"
TEST_IMG_DIR  = BASE_DIR / "test_images"

YOLO_WEIGHT_PATH = Path("/content/drive/MyDrive/ai_project_data/raw/sprint_ai_project1_data/yolo_dataset/yolo_weights/best_yolo11m.pt")
RTDETR_WEIGHT_PATH = Path("/content/drive/MyDrive/ai_project_data/raw/sprint_ai_project1_data/yolo_dataset/detr_weights/best_rtdetr.pt")
WORK_DIR = BASE_DIR / "detector_classifier_pipeline"
WORK_DIR.mkdir(parents=True, exist_ok=True)

CROP_ROOT = WORK_DIR / "cls_crops"
GT_TRAIN_DIR = CROP_ROOT / "gt_train"
GT_VAL_DIR   = CROP_ROOT / "gt_val"
HARD_TRAIN_DIR = CROP_ROOT / "hard_train"
HARD_VAL_DIR   = CROP_ROOT / "hard_val"

for d in [GT_TRAIN_DIR, GT_VAL_DIR, HARD_TRAIN_DIR, HARD_VAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

def load_json(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

def collect_json_files(root_dir):
    return sorted(list(root_dir.rglob("*.json")))

def clamp(x, lo, hi):
    return max(lo, min(x, hi))

def bbox_xywh_to_xyxy(x, y, w, h):
    return [x, y, x + w, y + h]

def bbox_xyxy_to_xywh(x1, y1, x2, y2):
    return [x1, y1, x2 - x1, y2 - y1]

def clean_bbox_xywh(bbox, img_w, img_h, min_size=2):
    x, y, w, h = bbox
    x1, y1, x2, y2 = bbox_xywh_to_xyxy(x, y, w, h)

    x1 = clamp(x1, 0, img_w - 1)
    y1 = clamp(y1, 0, img_h - 1)
    x2 = clamp(x2, 0, img_w - 1)
    y2 = clamp(y2, 0, img_h - 1)

    if x2 <= x1 or y2 <= y1:
        return None
    if (x2 - x1) < min_size or (y2 - y1) < min_size:
        return None

    return [float(x1), float(y1), float(x2), float(y2)]

def expand_bbox_xyxy(box, img_w, img_h, pad_ratio=0.12):
    x1, y1, x2, y2 = box
    bw = x2 - x1
    bh = y2 - y1

    pad_w = bw * pad_ratio
    pad_h = bh * pad_ratio

    nx1 = clamp(int(round(x1 - pad_w)), 0, img_w - 1)
    ny1 = clamp(int(round(y1 - pad_h)), 0, img_h - 1)
    nx2 = clamp(int(round(x2 + pad_w)), 0, img_w - 1)
    ny2 = clamp(int(round(y2 + pad_h)), 0, img_h - 1)

    if nx2 <= nx1 or ny2 <= ny1:
        return [int(x1), int(y1), int(x2), int(y2)]
    return [nx1, ny1, nx2, ny2]

def compute_iou_xyxy(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_w = max(0.0, x2 - x1)
    inter_h = max(0.0, y2 - y1)
    inter = inter_w * inter_h

    area1 = max(0.0, box1[2] - box1[0]) * max(0.0, box1[3] - box1[1])
    area2 = max(0.0, box2[2] - box2[0]) * max(0.0, box2[3] - box2[1])
    union = area1 + area2 - inter + 1e-9

    return inter / union

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

In [ ]:
json_files = collect_json_files(TRAIN_ANN_DIR)
print("json files:", len(json_files))

category_ids = set()
for jf in json_files:
    data = load_json(jf)
    for cat in data.get("categories", []):
        category_ids.add(cat["id"])

category_ids = sorted(category_ids)

cat_id_to_label = {cat_id: idx for idx, cat_id in enumerate(category_ids)}
label_to_cat_id = {idx: cat_id for cat_id, idx in cat_id_to_label.items()}

num_classes = len(category_ids)
print("num_classes:", num_classes)
print("sample mapping:", list(cat_id_to_label.items())[:10])

json files: 763
num_classes: 56
sample mapping: [(1900, 0), (2483, 1), (3351, 2), (3483, 3), (3544, 4), (3743, 5), (3832, 6), (4543, 7), (12081, 8), (12247, 9)]


In [ ]:
all_records = []

for jf in json_files:
    data = load_json(jf)

    if len(data.get("images", [])) == 0:
        continue

    img_info = data["images"][0]
    file_name = img_info["file_name"]
    width = img_info["width"]
    height = img_info["height"]

    objs = []
    for ann in data.get("annotations", []):
        cat_id = ann["category_id"]
        bbox = ann["bbox"]

        clean_box = clean_bbox_xywh(bbox, width, height)
        if clean_box is None:
            continue

        objs.append({
            "category_id": cat_id,
            "label": cat_id_to_label[cat_id],
            "bbox_xyxy": clean_box
        })

    if len(objs) == 0:
        continue

    all_records.append({
        "file_name": file_name,
        "width": width,
        "height": height,
        "objects": objs
    })

print("usable images:", len(all_records))

# =========================
# 3. train / val split
#    rare class를 고려해서
#    - train: rare class 최소 2장
#    - val:   rare class 가능하면 최소 1장
# =========================
val_ratio = 0.2
MIN_TRAIN_IMAGES_PER_RARE_CLASS = 2
MIN_VAL_IMAGES_PER_RARE_CLASS = 1
RARE_IMAGE_THRESHOLD = 8

# 클래스 -> 해당 클래스를 포함하는 이미지 인덱스들
class_to_indices = defaultdict(list)
for i, rec in enumerate(all_records):
    cls_set = {ann["category_id"] for ann in rec["objects"]}
    for cls in cls_set:
        class_to_indices[cls].append(i)

# rare class 정의
rare_classes_for_split = {
    cls for cls, idxs in class_to_indices.items()
    if len(idxs) <= RARE_IMAGE_THRESHOLD
}

print(f"rare classes for split ({len(rare_classes_for_split)}):",
      sorted(list(rare_classes_for_split))[:20], "...")

num_images = len(all_records)
target_train_size = int(num_images * (1 - val_ratio))
target_val_size = num_images - target_train_size

all_indices = set(range(num_images))
train_idx_set = set()
val_idx_set = set()

# --------------------------------------------------
# 1) rare class별로 train / val 최소 개수 먼저 확보
# --------------------------------------------------
for cls in sorted(rare_classes_for_split):
    idxs = class_to_indices[cls][:]
    random.shuffle(idxs)
    total = len(idxs)

    if total == 1:
        need_train = 1
        need_val = 0
    elif total == 2:
        need_train = 1
        need_val = 1
    else:
        need_train = min(MIN_TRAIN_IMAGES_PER_RARE_CLASS, total - 1)
        need_val = min(MIN_VAL_IMAGES_PER_RARE_CLASS, total - need_train)

    # val 먼저 확보
    val_added = 0
    for idx in idxs:
        if val_added >= need_val:
            break
        if idx not in train_idx_set and idx not in val_idx_set:
            val_idx_set.add(idx)
            val_added += 1

    # train 확보
    train_added = 0
    for idx in idxs:
        if train_added >= need_train:
            break
        if idx not in train_idx_set and idx not in val_idx_set:
            train_idx_set.add(idx)
            train_added += 1

    # multi-label 겹침 보정
    current_train = sum(1 for idx in idxs if idx in train_idx_set)
    current_val = sum(1 for idx in idxs if idx in val_idx_set)

    if current_train < need_train:
        for idx in idxs:
            if current_train >= need_train:
                break
            if idx in val_idx_set:
                val_idx_set.remove(idx)
                train_idx_set.add(idx)
                current_train += 1
                current_val -= 1

    if current_val < need_val:
        for idx in idxs:
            if current_val >= need_val:
                break
            if idx in train_idx_set:
                train_count_after_remove = sum(
                    1 for j in idxs if j in train_idx_set and j != idx
                )
                if train_count_after_remove >= need_train:
                    train_idx_set.remove(idx)
                    val_idx_set.add(idx)
                    current_train -= 1
                    current_val += 1
            elif idx not in val_idx_set:
                val_idx_set.add(idx)
                current_val += 1

# --------------------------------------------------
# 2) 전체 target size 맞추기
# --------------------------------------------------
remaining_indices = list(all_indices - train_idx_set - val_idx_set)
random.shuffle(remaining_indices)

need_val_more = max(0, target_val_size - len(val_idx_set))
val_fill = remaining_indices[:need_val_more]
val_idx_set.update(val_fill)

remaining_indices = [i for i in remaining_indices if i not in val_idx_set]

train_idx_set.update(remaining_indices)

# train이 target보다 작으면 val에서 일부 이동
if len(train_idx_set) < target_train_size:
    movable_from_val = list(val_idx_set)
    random.shuffle(movable_from_val)

    for idx in movable_from_val:
        if len(train_idx_set) >= target_train_size:
            break

        rec = all_records[idx]   # ✅ 수정
        classes_in_img = {ann["category_id"] for ann in rec["objects"]}

        can_move = True
        for cls in classes_in_img:
            idxs = class_to_indices[cls]
            total = len(idxs)

            if cls in rare_classes_for_split:
                if total == 1:
                    min_val_needed = 0
                elif total == 2:
                    min_val_needed = 1
                else:
                    min_val_needed = MIN_VAL_IMAGES_PER_RARE_CLASS
            else:
                min_val_needed = 0

            current_val_count = sum(1 for j in idxs if j in val_idx_set)
            if current_val_count - 1 < min_val_needed:
                can_move = False
                break

        if can_move:
            val_idx_set.remove(idx)
            train_idx_set.add(idx)

# train이 너무 많으면 val로 이동
if len(train_idx_set) > target_train_size:
    movable_from_train = list(train_idx_set)
    random.shuffle(movable_from_train)

    for idx in movable_from_train:
        if len(train_idx_set) <= target_train_size:
            break

        rec = all_records[idx]   # ✅ 수정
        classes_in_img = {ann["category_id"] for ann in rec["objects"]}

        can_move = True
        for cls in classes_in_img:
            idxs = class_to_indices[cls]
            total = len(idxs)

            if cls in rare_classes_for_split:
                if total == 1:
                    min_train_needed = 1
                elif total == 2:
                    min_train_needed = 1
                else:
                    min_train_needed = MIN_TRAIN_IMAGES_PER_RARE_CLASS
            else:
                min_train_needed = 0

            current_train_count = sum(1 for j in idxs if j in train_idx_set)
            if current_train_count - 1 < min_train_needed:
                can_move = False
                break

        if can_move:
            train_idx_set.remove(idx)
            val_idx_set.add(idx)

# 최종 인덱스 정리
train_indices = sorted(train_idx_set)
val_indices = sorted(val_idx_set)

assert len(set(train_indices) & set(val_indices)) == 0
assert len(train_indices) + len(val_indices) == num_images

train_records = [all_records[i] for i in train_indices]
val_records   = [all_records[i] for i in val_indices]

print(f"train: {len(train_records)}")
print(f"val: {len(val_records)}")

# split 검증 출력
train_class_image_freq = Counter()
val_class_image_freq = Counter()

for rec in train_records:
    for cls in {ann["category_id"] for ann in rec["objects"]}:
        train_class_image_freq[cls] += 1

for rec in val_records:
    for cls in {ann["category_id"] for ann in rec["objects"]}:
        val_class_image_freq[cls] += 1

print("rare class coverage check:")
for cls in sorted(rare_classes_for_split):
    total_img_cnt = len(class_to_indices[cls])
    train_img_cnt = train_class_image_freq[cls]
    val_img_cnt = val_class_image_freq[cls]

    if total_img_cnt == 1:
        target_train = 1
        target_val = 0
    elif total_img_cnt == 2:
        target_train = 1
        target_val = 1
    else:
        target_train = min(MIN_TRAIN_IMAGES_PER_RARE_CLASS, total_img_cnt - 1)
        target_val = min(MIN_VAL_IMAGES_PER_RARE_CLASS, total_img_cnt - target_train)

    print(
        f"  class {cls}: total={total_img_cnt}, "
        f"train={train_img_cnt}, val={val_img_cnt}, "
        f"target_train>={target_train}, target_val>={target_val}"
    )

def save_gt_crops(records, out_dir, pad_ratio=0.12):
    out_dir = Path(out_dir)
    ensure_dir(out_dir)   # ✅ 추가
    count = 0
    missing_count = 0
    open_error_count = 0

    for rec in tqdm(records):
        img_path = TRAIN_IMG_DIR / rec["file_name"]
        if not img_path.exists():
            missing_count += 1
            continue

        try:
            img = Image.open(img_path).convert("RGB")
        except Exception:
            open_error_count += 1
            continue

        img_w, img_h = img.size
        stem = Path(rec["file_name"]).stem

        for idx, obj in enumerate(rec["objects"]):
            label = obj["label"]
            box = expand_bbox_xyxy(obj["bbox_xyxy"], img_w, img_h, pad_ratio)

            x1, y1, x2, y2 = box
            if x2 <= x1 or y2 <= y1:
                continue

            crop = img.crop((x1, y1, x2, y2))

            class_dir = out_dir / str(label)
            ensure_dir(class_dir)

            save_path = class_dir / f"{stem}_gt_{idx}.png"
            crop.save(save_path)
            count += 1

    print(f"[{out_dir.name}] saved={count}, missing={missing_count}, open_error={open_error_count}")
    return count

print("saving GT train crops...")
n1 = save_gt_crops(train_records, GT_TRAIN_DIR, pad_ratio=0.12)

print("saving GT val crops...")
n2 = save_gt_crops(val_records, GT_VAL_DIR, pad_ratio=0.12)

print("GT train crops:", n1)
print("GT val crops:", n2)

usable images: 762
rare classes for split (25): [3544, 3743, 4543, 12081, 12247, 12778, 13395, 16551, 16688, 19552, 19607, 19861, 21771, 22362, 24850, 25438, 27777, 27926, 27993, 29345] ...
train: 609
val: 153
rare class coverage check:
  class 3544: total=6, train=4, val=2, target_train>=2, target_val>=1
  class 3743: total=3, train=2, val=1, target_train>=2, target_val>=1
  class 4543: total=5, train=2, val=3, target_train>=2, target_val>=1
  class 12081: total=3, train=2, val=1, target_train>=2, target_val>=1
  class 12247: total=3, train=2, val=1, target_train>=2, target_val>=1
  class 12778: total=6, train=4, val=2, target_train>=2, target_val>=1
  class 13395: total=3, train=2, val=1, target_train>=2, target_val>=1
  class 16551: total=3, train=2, val=1, target_train>=2, target_val>=1
  class 16688: total=8, train=4, val=4, target_train>=2, target_val>=1
  class 19552: total=3, train=2, val=1, target_train>=2, target_val>=1
  class 19607: total=6, train=5, val=1, target_train>=2,

100%|██████████| 609/609 [07:32<00:00,  1.35it/s]


[gt_train] saved=609, missing=0, open_error=0
saving GT val crops...


100%|██████████| 153/153 [01:22<00:00,  1.86it/s]

[gt_val] saved=153, missing=0, open_error=0
GT train crops: 609
GT val crops: 153


In [31]:
def save_crop(img, box_xyxy, save_path, pad_ratio=0.12):
    img_w, img_h = img.size
    box = expand_bbox_xyxy(box_xyxy, img_w, img_h, pad_ratio)
    x1, y1, x2, y2 = box
    crop = img.crop((x1, y1, x2, y2))
    crop.save(save_path)

def build_confusion_and_hard_samples(
    model,
    records,
    img_dir,
    hard_out_dir,
    iou_match_thr=0.5,
    pred_conf_thr=0.03,
    save_missed_gt=True
):
    confusion = np.zeros((num_classes, num_classes), dtype=np.int64)
    pair_counter = Counter()

    matched_correct = 0
    matched_wrong = 0
    missed_gt = 0
    total_gt = 0

    hard_rows = []

    for rec in tqdm(records):
        file_name = rec["file_name"]
        img_path = img_dir / file_name
        if not img_path.exists():
            continue

        try:
            img = Image.open(img_path).convert("RGB")
        except Exception:
            continue

        results = model.predict(
            source=str(img_path),
            conf=pred_conf_thr,
            verbose=False,
            save=False
        )

        result = results[0]

        gt_objects = rec["objects"]
        total_gt += len(gt_objects)

        pred_boxes = []
        pred_labels = []
        pred_scores = []

        if result.boxes is not None and len(result.boxes) > 0:
            pred_boxes = result.boxes.xyxy.cpu().numpy().tolist()
            pred_labels = result.boxes.cls.cpu().numpy().astype(int).tolist()
            pred_scores = result.boxes.conf.cpu().numpy().tolist()

        used_pred = set()
        stem = Path(file_name).stem

        for gt_idx, gt in enumerate(gt_objects):
            gt_label = gt["label"]
            gt_box = gt["bbox_xyxy"]

            best_iou = 0.0
            best_pred_idx = -1

            for pi, pbox in enumerate(pred_boxes):
                if pi in used_pred:
                    continue
                iou = compute_iou_xyxy(gt_box, pbox)
                if iou > best_iou:
                    best_iou = iou
                    best_pred_idx = pi

            if best_pred_idx >= 0 and best_iou >= iou_match_thr:
                used_pred.add(best_pred_idx)
                pred_label = pred_labels[best_pred_idx]
                pred_score = pred_scores[best_pred_idx]
                pred_box = pred_boxes[best_pred_idx]

                confusion[gt_label, pred_label] += 1
                pair_counter[(gt_label, pred_label)] += 1

                if gt_label == pred_label:
                    matched_correct += 1
                else:
                    matched_wrong += 1

                    class_dir = Path(hard_out_dir) / str(gt_label)
                    ensure_dir(class_dir)

                    save_path = class_dir / f"{stem}_wrong_gt{gt_label}_pred{pred_label}_iou{best_iou:.3f}_conf{pred_score:.3f}.png"
                    save_crop(img, pred_box, save_path, pad_ratio=0.12)

                    hard_rows.append({
                        "file_name": file_name,
                        "type": "wrong_class",
                        "gt_label": gt_label,
                        "pred_label": pred_label,
                        "iou": best_iou,
                        "pred_score": pred_score,
                        "save_path": str(save_path)
                    })

            else:
                missed_gt += 1

                if save_missed_gt:
                    class_dir = Path(hard_out_dir) / str(gt_label)
                    ensure_dir(class_dir)

                    save_path = class_dir / f"{stem}_missed_gt{gt_label}_{gt_idx}.png"
                    save_crop(img, gt_box, save_path, pad_ratio=0.12)

                    hard_rows.append({
                        "file_name": file_name,
                        "type": "missed_gt",
                        "gt_label": gt_label,
                        "pred_label": -1,
                        "iou": 0.0,
                        "pred_score": 0.0,
                        "save_path": str(save_path)
                    })

    summary = {
        "total_gt": total_gt,
        "matched_correct": matched_correct,
        "matched_wrong": matched_wrong,
        "missed_gt": missed_gt
    }

    hard_df = pd.DataFrame(hard_rows)
    pair_df = pd.DataFrame([
        {"gt_label": g, "pred_label": p, "count": c}
        for (g, p), c in pair_counter.items()
    ]).sort_values("count", ascending=False)

    return confusion, pair_df, hard_df, summary

model = RTDETR(str(RTDETR_WEIGHT_PATH))

confusion, pair_df, hard_df, summary = build_confusion_and_hard_samples(
    model=model,
    records=val_records,
    img_dir=TRAIN_IMG_DIR,
    hard_out_dir=HARD_TRAIN_DIR,
    iou_match_thr=0.5,
    pred_conf_thr=0.03,
    save_missed_gt=True
)

print(summary)
print(pair_df.head(20))
hard_df.to_csv(WORK_DIR / "hard_samples.csv", index=False)
pair_df.to_csv(WORK_DIR / "confusion_pairs.csv", index=False)
np.save(WORK_DIR / "confusion_matrix.npy", confusion)

100%|██████████| 153/153 [00:18<00:00,  8.25it/s]

{'total_gt': 153, 'matched_correct': 149, 'matched_wrong': 1, 'missed_gt': 3}
    gt_label  pred_label  count
20         2           2     21
34        52          52      8
39         3           3      7
22        53          53      7
19         6           6      6
30        14          14      5
28        55          55      5
0          0           0      5
49        33          33      4
25        45          45      4
32        17          17      4
24        13          13      4
29        18          18      3
43        40          40      3
38        25          25      3
27        50          50      3
16        34          34      3
2         15          15      3
53         7           7      3
13        10          10      2


In [32]:
conf_df = pd.DataFrame(
    confusion,
    index=[label_to_cat_id[i] for i in range(num_classes)],
    columns=[label_to_cat_id[i] for i in range(num_classes)]
)

conf_df.to_csv(WORK_DIR / "confusion_matrix_readable.csv")
conf_df.iloc[:10, :10]

top_pairs = pair_df[pair_df["gt_label"] != pair_df["pred_label"]].copy()
top_pairs["gt_cat_id"] = top_pairs["gt_label"].map(label_to_cat_id)
top_pairs["pred_cat_id"] = top_pairs["pred_label"].map(label_to_cat_id)

top_pairs = top_pairs.sort_values("count", ascending=False)
top_pairs.head(30)

confusing_labels = set(top_pairs.head(20)["gt_label"].tolist()) | set(top_pairs.head(20)["pred_label"].tolist())
print("num confusing labels:", len(confusing_labels))
print(sorted(confusing_labels)[:30])

num confusing labels: 2
[36, 52]


In [33]:
class CropDataset(Dataset):
    def __init__(self, roots, transform=None):
        self.transform = transform
        self.samples = []

        if isinstance(roots, (str, Path)):
            roots = [roots]

        for root in roots:
            root = Path(root)
            if not root.exists():
                continue

            class_dirs = [p for p in root.iterdir() if p.is_dir()]
            class_dirs = sorted(class_dirs, key=lambda x: int(x.name))

            for class_dir in class_dirs:
                label = int(class_dir.name)
                for img_path in class_dir.glob("*"):
                    self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")

        if self.transform is not None:
            img = self.transform(img)

        return img, label

In [34]:
from torch.utils.data import WeightedRandomSampler

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(340, scale=(0.9, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomRotation(7),
    transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.05, hue=0.02),
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=3)
    ], p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize((340, 340)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

cls_train_ds = CropDataset([GT_TRAIN_DIR, HARD_TRAIN_DIR], transform=train_tf)
cls_val_ds   = CropDataset([GT_VAL_DIR], transform=val_tf)

print("cls_train_ds:", len(cls_train_ds))
print("cls_val_ds:", len(cls_val_ds))

train_labels = [label for _, label in cls_train_ds.samples]
label_counts = Counter(train_labels)

sample_weights = [1.0 / (label_counts[label] ** 0.5) for label in train_labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(cls_train_ds, batch_size=32, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(cls_val_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

cls_train_ds: 810
cls_val_ds: 153


In [35]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from tqdm import tqdm

NUM_CLASSES = 56

def build_classifier(num_classes=56, pretrained=True):
    if pretrained:
        weights = models.ConvNeXt_Small_Weights.DEFAULT
    else:
        weights = None

    model = models.convnext_small(weights=weights)
    in_features = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(in_features, num_classes)
    return model

# 모델 생성
cls_model = build_classifier(NUM_CLASSES).to(device)
num_epochs=15
# loss / optimizer / scheduler
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.AdamW(cls_model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total = 0
    correct = 0

    for images, labels in tqdm(loader, leave=False):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

    return total_loss / total, correct / total

@torch.no_grad()
def eval_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total = 0
    correct = 0

    for images, labels in tqdm(loader, leave=False):
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

    return total_loss / total, correct / total

CLS_SAVE_PATH = WORK_DIR / "best_classifier_convnext.pth"
best_val_acc = 0.0

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(cls_model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = eval_one_epoch(cls_model, val_loader, criterion, device)
    scheduler.step()

    print(
        f"[Epoch {epoch+1}] "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
    )

    if val_acc >= best_val_acc:
        best_val_acc = val_acc
        torch.save(cls_model.state_dict(), CLS_SAVE_PATH)
        print("✅ saved:", CLS_SAVE_PATH)

print("best_val_acc:", best_val_acc)

# best 모델 로드
cls_model = build_classifier(NUM_CLASSES).to(device)
cls_model.load_state_dict(torch.load(CLS_SAVE_PATH, map_location=device))
cls_model.eval()

# inference transform
cls_tf = transforms.Compose([
    transforms.Resize((340, 340)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

@torch.no_grad()
def classify_crop(pil_crop, model, transform, device):
    x = transform(pil_crop).unsqueeze(0).to(device)
    logits = model(x)
    probs = torch.softmax(logits, dim=1)[0]
    pred_label = int(torch.argmax(probs).item())
    pred_prob = float(probs[pred_label].item())
    return pred_label, pred_prob, probs.cpu().numpy()

top_confusing_pairs = top_pairs.head(30).copy()
confusing_label_set = set(top_confusing_pairs["gt_label"].tolist()) | set(top_confusing_pairs["pred_label"].tolist())

print("confusing label set size:", len(confusing_label_set))
print(sorted(list(confusing_label_set))[:50])

[Epoch 1] train_loss=3.2669 train_acc=0.3000 | val_loss=1.7710 val_acc=0.7974
✅ saved: /content/drive/MyDrive/ai_project_data/raw/sprint_ai_project1_data/detector_classifier_pipeline/best_classifier_convnext.pth


[Epoch 2] train_loss=1.5157 train_acc=0.8457 | val_loss=0.7248 val_acc=0.9739
✅ saved: /content/drive/MyDrive/ai_project_data/raw/sprint_ai_project1_data/detector_classifier_pipeline/best_classifier_convnext.pth


[Epoch 3] train_loss=0.7004 train_acc=0.9827 | val_loss=0.4963 val_acc=0.9935
✅ saved: /content/drive/MyDrive/ai_project_data/raw/sprint_ai_project1_data/detector_classifier_pipeline/best_classifier_convnext.pth


[Epoch 4] train_loss=0.5032 train_acc=0.9951 | val_loss=0.4611 val_acc=0.9935
✅ saved: /content/drive/MyDrive/ai_project_data/raw/sprint_ai_project1_data/detector_classifier_pipeline/best_classifier_convnext.pth


[Epoch 5] train_loss=0.4684 train_acc=0.9951 | val_loss=0.4484 val_acc=0.9935
✅ saved: /content/drive/MyDrive/ai_project_data/raw/sprint_ai_project1_data/detector_classifier_pipeline/best_classifier_convnext.pth


...

[Epoch 11] train_loss=0.4397 train_acc=0.9914 | val_loss=0.4384 val_acc=1.0000
✅ saved: /content/drive/MyDrive/ai_project_data/raw/sprint_ai_project1_data/detector_classifier_pipeline/best_classifier_convnext.pth


[Epoch 12] train_loss=0.4208 train_acc=0.9988 | val_loss=0.4374 val_acc=1.0000
✅ saved: /content/drive/MyDrive/ai_project_data/raw/sprint_ai_project1_data/detector_classifier_pipeline/best_classifier_convnext.pth


[Epoch 13] train_loss=0.4288 train_acc=0.9951 | val_loss=0.4393 val_acc=1.0000
✅ saved: /content/drive/MyDrive/ai_project_data/raw/sprint_ai_project1_data/detector_classifier_pipeline/best_classifier_convnext.pth


[Epoch 14] train_loss=0.4210 train_acc=0.9975 | val_loss=0.4391 val_acc=1.0000
✅ saved: /content/drive/MyDrive/ai_project_data/raw/sprint_ai_project1_data/detector_classifier_pipeline/best_classifier_convnext.pth


[Epoch 15] train_loss=0.4235 train_acc=0.9963 | val_loss=0.4389 val_acc=1.0000
✅ saved: /content/drive/MyDrive/ai_project_data/raw/sprint_ai_project1_data/detector_classifier_pipeline/best_classifier_convnext.pth
best_val_acc: 1.0
confusing label set size: 2
[36, 52]


In [36]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import torch

def bbox_xyxy_to_xywh(x1, y1, x2, y2):
    return [x1, y1, x2 - x1, y2 - y1]

def expand_bbox_xyxy(box, img_w, img_h, pad_ratio=0.08):
    x1, y1, x2, y2 = box
    w = x2 - x1
    h = y2 - y1

    pad_w = w * pad_ratio
    pad_h = h * pad_ratio

    nx1 = max(0, int(x1 - pad_w))
    ny1 = max(0, int(y1 - pad_h))
    nx2 = min(img_w - 1, int(x2 + pad_w))
    ny2 = min(img_h - 1, int(y2 + pad_h))
    return [nx1, ny1, nx2, ny2]

@torch.no_grad()
def classify_crop(pil_crop, model, transform, device):
    x = transform(pil_crop).unsqueeze(0).to(device)
    logits = model(x)
    probs = torch.softmax(logits, dim=1)[0]
    pred_label = int(torch.argmax(probs).item())
    pred_prob = float(probs[pred_label].item())
    return pred_label, pred_prob, probs.cpu().numpy()

def run_detector_plus_classifier_submission_csv(
    det_model,
    cls_model,
    TEST_IMAGE_DIR,
    label_to_cat_id,
    confusing_label_set,
    cls_tf,
    device,
    det_conf=0.15,
    crop_pad_ratio=0.08,
    cls_prob_thr=0.60,
    disagree_cls_prob_thr=0.75,
    final_score_thr=0.12,
    max_boxes_per_image=15,
    save_path=None,
):
    if save_path is None:
        save_path = Path("submission_detector_plus_classifier.csv")
    else:
        save_path = Path(save_path)

    rows = []
    annotation_id = 1

    results = det_model.predict(
        source=str(TEST_IMAGE_DIR),
        conf=det_conf,
        verbose=False,
        save=False
    )

    for output in results:
        file_name = os.path.basename(output.path)
        image_id = int(Path(file_name).stem)

        try:
            img = Image.open(output.path).convert("RGB")
        except Exception:
            continue

        img_w, img_h = img.size

        if output.boxes is None or len(output.boxes) == 0:
            continue

        boxes = output.boxes.xyxy.cpu().numpy()
        det_scores = output.boxes.conf.cpu().numpy()
        det_labels = output.boxes.cls.cpu().numpy().astype(int)

        order = np.argsort(-det_scores)
        if max_boxes_per_image is not None:
            order = order[:max_boxes_per_image]

        boxes = boxes[order]
        det_scores = det_scores[order]
        det_labels = det_labels[order]

        for box, det_score, det_label in zip(boxes, det_scores, det_labels):
            x1, y1, x2, y2 = box.tolist()

            x1 = max(0.0, min(x1, img_w - 1))
            y1 = max(0.0, min(y1, img_h - 1))
            x2 = max(0.0, min(x2, img_w - 1))
            y2 = max(0.0, min(y2, img_h - 1))

            if x2 <= x1 or y2 <= y1:
                continue

            det_box = [x1, y1, x2, y2]
            final_label = int(det_label)
            final_score = float(det_score)

            if int(det_label) in confusing_label_set:
                crop_box = expand_bbox_xyxy(det_box, img_w, img_h, crop_pad_ratio)
                cx1, cy1, cx2, cy2 = crop_box

                if cx2 > cx1 and cy2 > cy1:
                    crop = img.crop((cx1, cy1, cx2, cy2))
                    cls_label, cls_prob, cls_probs = classify_crop(crop, cls_model, cls_tf, device)

                    cls_det_prob = float(cls_probs[det_label])
                    margin = float(cls_prob - cls_det_prob)

                    if cls_label == det_label:
                        final_label = det_label
                        if cls_prob >= cls_prob_thr:
                            final_score = det_score * (0.65 + 0.35 * cls_prob)
                        else:
                            final_score = det_score * (0.85 + 0.15 * cls_prob)
                    else:
                        if cls_prob >= disagree_cls_prob_thr and margin >= 0.15 and det_score < 0.75:
                            final_label = cls_label
                            final_score = det_score * (0.40 + 0.60 * cls_prob)
                        else:
                            final_label = det_label
                            final_score = det_score * (0.75 + 0.25 * cls_det_prob)

            if final_score < final_score_thr:
                continue

            cat_id = label_to_cat_id[final_label]
            x, y, w, h = bbox_xyxy_to_xywh(x1, y1, x2, y2)

            if w <= 0 or h <= 0:
                continue

            rows.append({
                "annotation_id": annotation_id,
                "image_id": int(image_id),
                "category_id": int(cat_id),
                "bbox_x": round(float(x), 4),
                "bbox_y": round(float(y), 4),
                "bbox_w": round(float(w), 4),
                "bbox_h": round(float(h), 4),
                "score": round(float(final_score), 6),
            })
            annotation_id += 1

    df = pd.DataFrame(rows, columns=[
        "annotation_id",
        "image_id",
        "category_id",
        "bbox_x",
        "bbox_y",
        "bbox_w",
        "bbox_h",
        "score"
    ])

    df.to_csv(save_path, index=False)
    print(f"✅ saved: {save_path}")
    print(df.head())
    print("num rows:", len(df))
    return df

In [37]:
import torch
from pathlib import Path

# =========================
# 1. 경로 설정
# =========================
TEST_IMG_DIR = Path("/content/drive/MyDrive/ai_project_data/raw/sprint_ai_project1_data/yolo_dataset/images/test")

SAVE_PATH = Path("/content/drive/MyDrive/ai_project_data/raw/sprint_ai_project1_data/submission.csv")

# =========================
# 2. 모델 로드
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# detector
det_model = RTDETR("/content/drive/MyDrive/ai_project_data/raw/sprint_ai_project1_data/yolo_dataset/detr_weights/best_rtdetr.pt")

# classifier
import torchvision.models as models
import torch.nn as nn

num_classes = 56

def build_classifier(num_classes=56):
    model = models.convnext_small(weights=models.ConvNeXt_Small_Weights.DEFAULT)
    in_features = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(in_features, num_classes)
    return model

cls_model = build_classifier(num_classes=56)
state_dict = torch.load(
    "/content/drive/MyDrive/ai_project_data/raw/sprint_ai_project1_data/detector_classifier_pipeline/best_classifier_convnext.pth",
    map_location=device
)

cls_model.load_state_dict(state_dict)
cls_model.to(device)
cls_model.eval()

# =========================
# 6. 실행
# =========================
df = run_detector_plus_classifier_submission_csv(
    det_model,
    cls_model,
    TEST_IMG_DIR,
    label_to_cat_id,
    confusing_label_set,
    val_tf,
    device,
    det_conf=0.2,
    crop_pad_ratio=0.08,
    cls_prob_thr=0.60,
    disagree_cls_prob_thr=0.70,
    final_score_thr=0.22,
    max_boxes_per_image=10,
    save_path = BASE_DIR / "submission_detector_plus_classifier.csv"
)

✅ saved: /content/drive/MyDrive/ai_project_data/raw/sprint_ai_project1_data/submission_detector_plus_classifier.csv
   annotation_id  image_id  category_id    bbox_x    bbox_y    bbox_w  \
0              1         1        29345  173.7835  745.1285  175.2616   
1              2         1        27926  600.4020  683.5901  252.4336   
2              3         1         1900  158.5062  252.7523  202.1512   
3              4         1        27733  556.5026   75.3865  394.4835   
4              5         1        27733  556.6339   74.5679  394.1011   

     bbox_h     score  
0  290.9659  0.886796  
1  474.6376  0.813698  
2  119.8108  0.750940  
3  402.1327  0.453309  
4  402.3605  0.452971  
num rows: 3377


In [ ]:
def run_detector_plus_classifier_val(
    det_model,
    cls_model,
    img_dir,
    confusing_label_set,
    cls_tf,
    device,
    det_conf=0.12,
    crop_pad_ratio=0.12,
    final_score_thr=0.22,
):
    import os
    from PIL import Image
    from tqdm import tqdm

    results = {}

    img_paths = list(img_dir.glob("*"))

    for img_path in tqdm(img_paths):
        img = Image.open(img_path).convert("RGB")
        img_w, img_h = img.size

        outputs = det_model.predict(
            source=str(img_path),
            conf=det_conf,
            verbose=False
        )[0]

        pred_boxes = []
        pred_labels = []
        pred_scores = []

        if outputs.boxes is not None:
            for box, cls, score in zip(
                outputs.boxes.xyxy.cpu(),
                outputs.boxes.cls.cpu(),
                outputs.boxes.conf.cpu()
            ):
                det_box = box.tolist()
                det_label = int(cls.item())
                det_score = float(score.item())

                final_label = det_label
                final_score = det_score

                # 🔥 classifier 보정
                if det_label in confusing_label_set:
                    crop_box = expand_bbox_xyxy(det_box, img_w, img_h, crop_pad_ratio)
                    cx1, cy1, cx2, cy2 = crop_box

                    if cx2 > cx1 and cy2 > cy1:
                        crop = img.crop((cx1, cy1, cx2, cy2))
                        cls_label, cls_prob, _ = classify_crop(
                            crop, cls_model, cls_tf, device
                        )

                        if cls_label == det_label:
                            final_label = cls_label
                            final_score = det_score * (0.5 + 0.5 * cls_prob)
                        else:
                            final_label = cls_label
                            final_score = det_score * cls_prob * 0.85

                if final_score < final_score_thr:
                    continue

                pred_boxes.append(det_box)
                pred_labels.append(final_label)
                pred_scores.append(final_score)

        results[img_path.stem] = {
            "boxes": pred_boxes,
            "labels": pred_labels,
            "scores": pred_scores
        }

    return results

In [ ]:
def load_yolo_gt(label_path, img_w, img_h):
    boxes = []
    labels = []

    if not label_path.exists():
        return [], []

    with open(label_path) as f:
        lines = f.readlines()

    for line in lines:
        cls, cx, cy, w, h = map(float, line.split())

        x1 = (cx - w/2) * img_w
        y1 = (cy - h/2) * img_h
        x2 = (cx + w/2) * img_w
        y2 = (cy + h/2) * img_h

        boxes.append([x1, y1, x2, y2])
        labels.append(int(cls))

    return boxes, labels

In [ ]:
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0, x2 - x1) * max(0, y2 - y1)

    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])

    union = area1 + area2 - inter + 1e-6
    return inter / union


def match_boxes(pred_boxes, pred_labels, pred_scores, gt_boxes, gt_labels, iou_thr=0.5):
    used = set()
    tp, fp = 0, 0

    order = sorted(range(len(pred_scores)), key=lambda i: -pred_scores[i])

    for i in order:
        p_box = pred_boxes[i]
        p_label = pred_labels[i]

        best_iou = 0
        best_j = -1

        for j in range(len(gt_boxes)):
            if j in used:
                continue
            if gt_labels[j] != p_label:
                continue

            iou = compute_iou(p_box, gt_boxes[j])

            if iou > best_iou:
                best_iou = iou
                best_j = j

        if best_iou >= iou_thr:
            tp += 1
            used.add(best_j)
        else:
            fp += 1

    fn = len(gt_boxes) - len(used)

    return tp, fp, fn

In [ ]:
# =========================
# VAL 평가
# =========================
VAL_IMG_DIR = Path("/content/drive/MyDrive/ai_project_data/raw/sprint_ai_project1_data/yolo_dataset/images/val")
VAL_LABEL_DIR = Path("/content/drive/MyDrive/ai_project_data/raw/sprint_ai_project1_data/yolo_dataset/labels/val")

val_preds = run_detector_plus_classifier_val(
    det_model=det_model,
    cls_model=cls_model,
    img_dir=VAL_IMG_DIR,
    confusing_label_set=confusing_label_set,
    cls_tf=cls_infer_tf,
    device=device,
    det_conf=0.05,
    crop_pad_ratio=0.08,
    final_score_thr=0.1,
)

total_tp, total_fp, total_fn = 0, 0, 0

for img_name, pred in val_preds.items():
    img_path = VAL_IMG_DIR / f"{img_name}.png"
    label_path = VAL_LABEL_DIR / f"{img_name}.txt"

    from PIL import Image
    img = Image.open(img_path)
    w, h = img.size

    gt_boxes, gt_labels = load_yolo_gt(label_path, w, h)

    tp, fp, fn = match_boxes(
        pred["boxes"],
        pred["labels"],
        pred["scores"],
        gt_boxes,
        gt_labels,
        iou_thr=0.5
    )

    total_tp += tp
    total_fp += fp
    total_fn += fn

precision = total_tp / (total_tp + total_fp + 1e-6)
recall    = total_tp / (total_tp + total_fn + 1e-6)
f1        = 2 * precision * recall / (precision + recall + 1e-6)

print("\n========== VAL RESULT ==========")
print(f"TP: {total_tp}")
print(f"FP: {total_fp}")
print(f"FN: {total_fn}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")

100%|██████████| 181/181 [01:03<00:00,  2.87it/s]



========== VAL RESULT ==========
TP: 161
FP: 453
FN: 19
Precision: 0.2622
Recall   : 0.8944
F1       : 0.4055


## WBF(Weighted Boxes Fusion)

In [ ]:
!pip install ensemble-boxes

import os
import gc
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from ultralytics import YOLO
from ensemble_boxes import weighted_boxes_fusion
import torch


# =========================================================
# 0. 경로 / 설정
# =========================================================
TEST_IMAGE_DIR = OUT_DIR / "images" / "test"
SUBMISSION_PATH = OUT_DIR / "submission_wbf_tta.csv"

MODEL_PATHS = [
    "/content/runs/detect/runs_yolo11/exp_yolo11m/weights/best.pt",
    "/content/runs/detect/runs_yolo26/exp_yolo26s_adamw/weights/best.pt",
    "/content/runs/detect/runs_yolo26/exp_yolo26s_musgd/weights/best.pt",
    "/content/runs/detect/runs_rtdetr/exp_rtdetr_l2/weights/best.pt",
]

MODEL_WEIGHTS = [0.6, 0.6, 0.6, 1.5]

WBF_IOU_THR = 0.55
WBF_SKIP_BOX_THR = 0.001
FINAL_SCORE_THR = 0.10

# OOM 방지용으로 먼저 낮춰서 시작
YOLO_IMGSZ = 1024
YOLO_CONF = 0.01
YOLO_IOU = 0.6
DEVICE = 0
USE_TTA = False

try:
    yolo_to_cat_id
except NameError:
    yolo_to_cat_id = {idx: cat_id for cat_id, idx in cat_id_to_yolo.items()}


# =========================================================
# 1. 유틸 함수
# =========================================================
def get_image_size(image_path):
    with Image.open(image_path) as img:
        return img.size  # (width, height)


def xyxy_to_normalized(boxes, width, height):
    boxes = np.asarray(boxes, dtype=np.float32).copy()
    if len(boxes) == 0:
        return []
    boxes[:, [0, 2]] /= float(width)
    boxes[:, [1, 3]] /= float(height)
    boxes = np.clip(boxes, 0.0, 1.0)
    return boxes.tolist()


def normalized_to_xyxy(boxes, width, height):
    boxes = np.asarray(boxes, dtype=np.float32).copy()
    if len(boxes) == 0:
        return []
    boxes[:, [0, 2]] *= float(width)
    boxes[:, [1, 3]] *= float(height)
    return boxes.tolist()


def clip_xyxy_boxes(boxes, width, height):
    boxes = np.asarray(boxes, dtype=np.float32).copy()
    if len(boxes) == 0:
        return boxes, np.array([], dtype=bool)

    boxes[:, 0] = np.clip(boxes[:, 0], 0, width - 1)
    boxes[:, 1] = np.clip(boxes[:, 1], 0, height - 1)
    boxes[:, 2] = np.clip(boxes[:, 2], 0, width - 1)
    boxes[:, 3] = np.clip(boxes[:, 3], 0, height - 1)

    valid = (boxes[:, 2] > boxes[:, 0]) & (boxes[:, 3] > boxes[:, 1])
    return boxes[valid], valid


def ultralytics_result_to_pred_dict(result, image_width, image_height):
    if result.boxes is None or len(result.boxes) == 0:
        return {"boxes": [], "scores": [], "labels": []}

    boxes = result.boxes.xyxy.detach().cpu().numpy().astype(np.float32)
    scores = result.boxes.conf.detach().cpu().numpy().astype(np.float32)
    labels = result.boxes.cls.detach().cpu().numpy().astype(int)

    boxes[:, 0] = np.clip(boxes[:, 0], 0, image_width - 1)
    boxes[:, 1] = np.clip(boxes[:, 1], 0, image_height - 1)
    boxes[:, 2] = np.clip(boxes[:, 2], 0, image_width - 1)
    boxes[:, 3] = np.clip(boxes[:, 3], 0, image_height - 1)

    valid = (boxes[:, 2] > boxes[:, 0]) & (boxes[:, 3] > boxes[:, 1])

    boxes = boxes[valid]
    scores = scores[valid]
    labels = labels[valid]

    return {
        "boxes": boxes.tolist(),
        "scores": scores.tolist(),
        "labels": labels.tolist()
    }


def run_wbf(predictions, image_width, image_height,
            weights=None, iou_thr=0.55, skip_box_thr=0.01):
    boxes_list = []
    scores_list = []
    labels_list = []

    for pred in predictions:
        boxes = pred["boxes"]
        scores = pred["scores"]
        labels = pred["labels"]

        if len(boxes) == 0:
            boxes_list.append([])
            scores_list.append([])
            labels_list.append([])
            continue

        norm_boxes = xyxy_to_normalized(boxes, image_width, image_height)
        boxes_list.append(norm_boxes)
        scores_list.append([float(s) for s in scores])
        labels_list.append([int(l) for l in labels])

    fused_boxes, fused_scores, fused_labels = weighted_boxes_fusion(
        boxes_list=boxes_list,
        scores_list=scores_list,
        labels_list=labels_list,
        weights=weights,
        iou_thr=iou_thr,
        skip_box_thr=skip_box_thr
    )

    fused_boxes = np.asarray(
        normalized_to_xyxy(fused_boxes, image_width, image_height),
        dtype=np.float32
    )
    fused_boxes, valid = clip_xyxy_boxes(fused_boxes, image_width, image_height)

    fused_scores = np.asarray(fused_scores, dtype=np.float32)[valid]
    fused_labels = np.asarray(fused_labels, dtype=np.int32)[valid]

    return {
        "boxes": fused_boxes.tolist(),
        "scores": fused_scores.tolist(),
        "labels": fused_labels.tolist()
    }


# =========================================================
# 2. 모델 로드
# =========================================================
gc.collect()
torch.cuda.empty_cache()

models = [YOLO(path) for path in MODEL_PATHS]


# =========================================================
# 3. test 이미지 목록
# =========================================================
image_paths = sorted([
    p for p in Path(TEST_IMAGE_DIR).iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
])

print(f"num test images: {len(image_paths)}")


# =========================================================
# 4. 이미지 1장씩 -> 각 모델 예측 -> WBF -> 저장
# =========================================================
rows = []
annotation_id = 1

with torch.no_grad():
  for img_idx, image_path in enumerate(image_paths):
    file_name = image_path.name
    image_id = int(Path(file_name).stem)
    img_w, img_h = get_image_size(image_path)

    predictions_for_this_image = []

    for model_idx, model in enumerate(models):
        results = model.predict(
            source=str(image_path),   # 리스트 전체 X, 한 장씩
            imgsz=YOLO_IMGSZ,
            batch=1,
            conf=YOLO_CONF,
            iou=YOLO_IOU,
            augment=USE_TTA,
            device=DEVICE,
            half=True,                # FP16
            verbose=False
        )

        result = results[0]
        pred = ultralytics_result_to_pred_dict(result, img_w, img_h)
        predictions_for_this_image.append(pred)

        # 메모리 정리
        del results
        del result

    fused = run_wbf(
        predictions=predictions_for_this_image,
        image_width=img_w,
        image_height=img_h,
        weights=MODEL_WEIGHTS,
        iou_thr=WBF_IOU_THR,
        skip_box_thr=WBF_SKIP_BOX_THR
    )

    boxes = fused["boxes"]
    scores = fused["scores"]
    labels = fused["labels"]

    for box, score, label in zip(boxes, scores, labels):
        x1, y1, x2, y2 = box
        x = float(x1)
        y = float(y1)
        w = float(x2 - x1)
        h = float(y2 - y1)

        if w <= 0 or h <= 0:
            continue
        if score < FINAL_SCORE_THR:
            continue

        category_id = int(yolo_to_cat_id[int(label)]) if yolo_to_cat_id is not None else int(label)

        rows.append({
            "annotation_id": annotation_id,
            "image_id": image_id,
            "category_id": category_id,
            "bbox_x": float(x),
            "bbox_y": float(y),
            "bbox_w": float(w),
            "bbox_h": float(h),
            "score": float(score)
        })
        annotation_id += 1

    if (img_idx + 1) % 50 == 0:
      gc.collect()
      torch.cuda.empty_cache()
      print(f"processed: {img_idx + 1}/{len(image_paths)}")

# =========================================================
# 5. 저장
# =========================================================
df = pd.DataFrame(rows)
df.to_csv(SUBMISSION_PATH, index=False)
print(f"saved to: {SUBMISSION_PATH}")
print(df.head())

In [ ]:
df.to_csv(SUBMISSION_PATH, index=False, encoding="utf-8-sig")
df.to_csv("/content/drive/MyDrive/submission.csv", index=False)

print(df.head())
print("saved: /content/drive/MyDrive/submission.csv")